Textual Cleaning of ECB and FED Stability Reports and MOnetary policy reports, effectively all teh main tet that I will not be labelling. 

In [6]:
# Importing the operating system interface, basically where I will interact with the file system. 
import os
# Used when I need to asscertain pattern matching and or text manipulation.
import re
# Provides access to command-line arguments
import sys
# The actual format the OLMO model will be able to read, Since everything that is getting fed through is acctually a Txt file. 
import json
# Filename pattern matching
import glob
# Command-line argument parser 
import argparse
# Cryptographic hashing
import hashlib
# Unicode character database
import unicodedata
# Fixes Text For You
import ftfy
# Encodes text into tokens
import tiktoken
# Path package
from pathlib import Path
# only needed for the Fred Speeches Conversion
import pandas as pd

File paths where my documents lie, alongisde configuration for my model and tokeniser

In [7]:
data_path = Path("/Users/lorenzouberti/Desktop/BANQUE DE FRANCE /Data Folders/FED BRANCH/POLICY DECISIONS/txt/")
INPUT_DIRS = [data_path]    
OUT_PATH = "fed_remaining_statements_unlabeled.jsonl"

# Configuration so that it is inline with the OLMO 2 modles
# Effetcively the max numbers of tokens that the model can process, for anything over it is handeled by splitting. 
TARGET_TOKENS = 2048
# For text that isn't relevant and ost likely boiler plates from a failed scrape
MIN_TOKENS = 64
# discards chunks shorter than this many characters
MIN_CHARS = 200
# discards chunks with a lower share of letters (tables / number dumps)
MIN_ALPHA = 0.60
# drop standalone 'Sources:/Notes:/Chart' furniture lines
DROP_NOTES = True

Extra Step for FOMC Statements

Extra Step Need for The FOMC minutes

In [8]:
#data = pd.read_csv(data_path / "fomc_sentences_remaining.csv")
#data.head(10)
#all_text = "\n".join(data["sentence"].dropna().astype(str))
#with open("fed_FOMC_remaining_sentences.txt", "w", encoding="utf-8") as f:
    #f.write(all_text)


Extra Step for Fed Speeches since they need to be first converted into txt files before running.

In [9]:
#data = pd.read_csv(data_path / "fed_speeches.csv")
#all_text = "\n".join(data["text"].dropna().astype(str))
#with open("fed_speeches_merged.txt", "w", encoding="utf-8") as f:
    #f.write(all_text)

Tokeniser

In [10]:
# The justifcation for picking this is since the cl100k_base
# shares the base vocab, the token counts will match that of the dolma2 tokenizer
enc = tiktoken.get_encoding("cl100k_base")
def n_tokens(s): return len(enc.encode(s))


Boiler Plate Cleaning

In [11]:
# The following function is designed to clean the boiler plate which can be found at the top and bottom of the majority of my
# scraped FED documents, alothough most documents were cleaned during the scraping phase, there still could be remenants that
# could potentially pose issues. 

# Alongside boiler plate issues, most notably for the Monetary, Financial Stability  Reports etc, there are many graphs which are outside the scope
# of this project, since in the intial scrape they were removed, yet their headers and footnotes may have been picked up, this therefore requires me to
# remove them from the text. 
def clean_text(text, drop_notes=True):
    # Unicode normalization is the initial step since I need to ensure that the text remains consistent and predictable throughout the whole document.
    # I used the ftfy (fixes text for you) library to fix the standard/common Unicode encoding issues. 
    text = ftfy.fix_text(text)
    # Normalizes Unicode characters to NFKC, which stands for Normalization Form Compatibility Composition. 
    text = unicodedata.normalize("NFKC", text)
    # Replaces non-breaking spaces with regular spaces
    text = text.replace("\u00a0", " ")
    # Removes zero-width spaces and the byte order mark
    text = re.sub(r"[\u200b-\u200f\ufeff]", "", text)
    # Replaces middle dots and bullet points with spaces
    text = re.sub(r"[\u00b7\u2022]+", " ", text)
    
    # Fix soft hyphens and line-break hyphens, effectively similar to teh standard convention which is seen in the PDF versions of the Market reports, may words at teh end of a
    # page contain hyphens such as the communication when at the end of a page will read communi - cation (new line) this is removed and accounted for. 
    text = re.sub(r"(\w+)\u00ad\s*\n?\s*(\w+)", r"\1\2", text)
    # Removes any remaining soft hyphens 
    text = re.sub(r"\u00ad", "", text)
    # Another pattern to join words with Unicode soft hyphens.
    text = re.sub(r"(\w+)­(\w+)", r"\1\2", text)
    # Joins words that were split by a hyphen at line breaks
    text = re.sub(r"(\w+)-\n\s*(\w+)", r"\1\2", text)
    # Joins words with hyphens followed by spaces
    text = re.sub(r"(\w+)-\s+(\w+)", r"\1\2", text)
    # Reconstruct fractions - "51⁄4" → "5 1/4"
    text = re.sub(r"(\d+)\s*[⁄/]\s*(\d+)", r"\1 \2", text)
    text = re.sub(r"(\d+)\s*[⁄/]\s*(\d+)", r"\1 \2", text)  # second pass for edge cases
    
    # REMOVE ALL BOILERPLATE - aggressive multi-pass, Using Artificial Intelligence Tools, I gave extracts of each year of all reports and told the LLM to extract all relevant boiler plates and graphs outlines. 
    boilerplate_patterns = [
        r"For use at \d{1,2}:\d{2} [ap]\.m\.,?\s*E\.?D\.?T\.?",
        r"Board of Governors of the Federal Reserve System",
        r"Monetary Policy Report to the Congress",
        r"Letter of Transmittal",
        r"THE PRESIDENT OF THE SENATE",
        r"THE SPEAKER OF THE HOUSE OF REPRESENTATIVES",
        r"Dear Mr\. President,?",
        r"Dear Madam Speaker,?",
        r"pursuant to section 2B of the Federal Reserve Act",
        r"Submitted pursuant to section 2B",
        r"Alan Greenspan, Chairman",
        r"Washington, D\.C\.,",
        r"February \d{1,2}, \d{4}",
        r"Wednesday",
        r"Report submitted to the Congress on",
        r"Sincerely,",
        r"Contents",
        r"^Page$",
        r"^\s*Page\s+\d+\s*$",
        r"^\s*\d+\s*$",
    ]
    for pattern in boilerplate_patterns:
        text = re.sub(pattern, "", text, flags=re.IGNORECASE | re.MULTILINE)
    
    # Remove ALL chart titles and figure labels, following the same protocol as above
    chart_patterns = [
        r"^Selected interest rates$",
        r"^Percent$", 
        r"^Ten-year Treasury$",
        r"^Two-year Treasury$",
        r"^Intended federal funds rate$",
        r"^Change in real GDP$",
        r"^Change in PCE chain-type price index$",
        r"^Change in real income and consumption$",
        r"^Wealth-to-income ratio$",
        r"^Change in house prices$",
        r"^Mortgage rates$",
        r"^Private housing starts$",
        r"^Household financial obligations ratio$",
        r"^Delinquency rates on selected types of household loans$",
        r"^Change in real business fixed investment$",
        r"^Change in real business inventories$",
        r"^Before-tax profits of nonfinancial corporations",
        r"^Selected components of net business financing",
        r"^Financing gap and net equity retirement",
        r"^Net percentage of domestic banks tightening",
        r"^Default rate on outstanding corporate bonds$",
        r"^Federal receipts and expenditures$",
        r"^Net saving$",
        r"^Change in real government expenditures",
        r"^Federal government debt held by the public$",
        r"^Treasury securities held by foreign investors",
        r"^State and local government net saving$",
        r"^U\.S\. trade and current account balances$",
        r"^Prices of oil and of nonfuel commodities$",
        r"^Prices of major nonfuel commodities$",
        r"^U\.S\. net financial inflows$",
        r"^U\.S\. net international securities transactions$",
        r"^Civilian unemployment rate$",
        r"^Net change in payroll employment$",
        r"^Labor force participation rate$",
        r"^Measures of change in hourly compensation$",
        r"^Change in PCE prices excluding food and energy$",
        r"^Alternative measures of price change$",
        r"^Change in consumer prices$",
        r"^TIPS-based inflation compensation$",
        r"^Change in unit labor costs$",
        r"^Interest rates on selected Treasury securities$",
        r"^Spreads of corporate bond yields over",
        r"^Implied S&P 500 volatility$",
        r"^Stock price indexes$",
        r"^Growth of domestic nonfinancial debt$",
        r"^M2 growth rate$",
        r"^Equity indexes in selected foreign industrial countries$",
        r"^Spread on internationally issued sovereign debt",
        r"^Official interest rates in selected foreign industrial countries$",
        r"^U\.S\. dollar nominal exchange rate",
        r"^U\.S\. dollar exchange rate against",
        r"^Equity indexes in selected emerging-market economies$",
        r"^NOTE\.",
        r"^SOURCE\.",
        r"^SOURCES\.",
        r"Figure\s+\d+",
        r"Chart\s+\d+",
        r"Table\s+\d+",
        r"Panel\s+[A-Z]",
        r"^\s*[A-Z][a-z]+day\s*$",  # Day names
        r"^\s*[A-Z][a-z]+day,\s+[A-Z][a-z]+\s+\d{1,2},\s+\d{4}\s*$",
        r"^\s*[0-9]+\s*[–-]\s*[0-9]+\s*$",
        r"^\s*[0-9/]+\s+[0-9/]+\s+[0-9/]+\s*$",
        r"^\s*[0-9]{4}\s+[0-9]{4}\s+[0-9]{4}\s*$",
        r"^\s*(?:[0-9]+\s+){3,}[0-9]+$",
        r"^\s*(?:[A-Z][a-z]+)\s+(?:[A-Z][a-z]+)\s+\d+\s*$",
    ]
    for pattern in chart_patterns:
        text = re.sub(pattern, "", text, flags=re.IGNORECASE | re.MULTILINE)
    
    # Line-level filtering, this entailed droping chart data lines. 
    # Splits text into lines and filters out empty ones.
    lines = text.split("\n")
    filtered = []
    for line in lines:
        line = line.strip()
        if not line:
            continue
        # Skips lines that are pure chart noise
        # Skips lines with only numbers, spaces, and punctuation.
        if re.search(r"^[\d\s/\.\-:,]+$", line):
            continue
        # skips lines with date-like patterns (e.g., "1/2 3/4 5/6")
        if re.search(r"^\s*(?:\d{1,2}/\d{1,2}\s*)+$", line):
            continue
        # Skips lines with multiple 4-digit numbers.
        if re.search(r"^\s*(?:\d{4}\s*)+$", line):
            continue
        # Skips lines with numbers preceded by + or - signs
        if re.search(r"^\s*(?:[+\-]\s*\d+\.?\d*\s*)+$", line):
            continue
        # Skips lines with only mathematical operators and parentheses
        if re.search(r"^[\s()+\-*/=]+$", line):
            continue
        # Skips lines with a single number.
        if re.search(r"^\s*[0-9]+\s*$", line):
            continue
        
        # Alpha ratio filter (aggressive for chart-heavy MPRs)
        # Filters lines based on letter-to-character ratio
        # Lines with very few letters, i.e less thant  15% letters are likely chart data or numerical artifacts.
        alpha_count = sum(1 for c in line if c.isalpha())
        total_count = len(line)
        if total_count > 0:
            if alpha_count < 3:
                continue
            # The isascii() check ensures this only applies to ASCII lines (English text)
            if line.isascii() and alpha_count / total_count < 0.15:
                continue
        
        filtered.append(line)
    
    text = "\n".join(filtered)
    
    # Remove remaining NOTE/SOURCE lines, These are often citations, footnotes, or references - not substantive content. Standalone numbers could be page numbers or footnote markers.
    if drop_notes:
        #  Removes lines starting with "NOTE:", "SOURCE:", "SOURCES:" and standalone numbers.
        text = re.sub(r"(?m)^\s*(?:NOTE|SOURCE|SOURCES|Note|Source)\s*[:.]?\s*.*$", "", text, flags=re.I)
        text = re.sub(r"(?m)^\s*[0-9]+\s*$", "", text)
    
    # Clean up whitespace and fix text artifacts, replaces multiple whitespace characters (spaces, tabs, formfeeds, carriage returns) with a single space.
    text = re.sub(r"[ \t\f\r]+", " ", text)
    # Removes spaces around newlines.
    text = re.sub(r" *\n *", "\n", text)
    # Replaces 3 or more consecutive newlines with 2.
    text = re.sub(r"\n{3,}", "\n\n", text)
    
    # Remove any boilerplate that got split across lines
    text = re.sub(r"(?m)^[A-Z][A-Z\s]+$", "", text)  
    
    return text.strip()

Chunking Phase

This packs the paragrphs into the tiokens, and where the mainsplitting occurs

In [12]:

# Creates a regex pattern to split text at sentence boundaries
# his pattern identifies sentence boundaries by looking for periods, exclamation points, 
# or question marks followed by whitespace. The (?<=) lookbehind ensures we split after the punctuation, keeping the punctuation with the sentence
_SENT = re.compile(r"(?<=[.!?])\s+")

# Splits text into paragraphs based on blank lines.
def _paras(text):
    return [p.strip() for p in re.split(r"\n\s*\n", text) if p.strip()]

# Main chincking functions
def chunk_text(text, target, min_tokens):
    chunks, buf, buf_tok = [], [], 0
    # Takes everything in the buffer and saves it as a chunk, then resets the buffer
    def flush():
        nonlocal buf, buf_tok
        if buf:
            chunks.append("\n\n".join(buf)); buf, buf_tok = [], 0
    for para in _paras(text):
        ptok = n_tokens(para)
        if ptok > target:                          # paragraph too big -> sentences
            flush()
            sbuf, stok = [], 0
            for sent in _SENT.split(para):
                m = n_tokens(sent)
                if m > target:                     # single sentence too big -> hard cut
                    if sbuf:
                        chunks.append(" ".join(sbuf)); sbuf, stok = [], 0
                    ids = enc.encode(sent)
                    for i in range(0, len(ids), target):
                        chunks.append(enc.decode(ids[i:i + target]))
                elif stok + m > target:
                    chunks.append(" ".join(sbuf)); sbuf, stok = [sent], m
                else:
                    sbuf.append(sent); stok += m
            if sbuf:
                chunks.append(" ".join(sbuf))
        elif buf_tok + ptok > target:
            flush(); buf, buf_tok = [para], ptok
        else:
            buf.append(para); buf_tok += ptok
    flush()
    return [c for c in chunks if n_tokens(c) >= min_tokens]

Cleaning Testing Start

In [13]:
sample = '''For use at 10:00 a.m., EST
Wednesday
February 16, 2005
Board of Governors of the Federal Reserve System
Monetary Policy Report to the Congress
February 16, 2005
Board of Governors of the Federal Reserve System
Monetary Policy Report to the Congress
Submitted pursuant to section 2B of the Federal Reserve Act
February 16, 2005
Letter of Transmittal
BOARD OF GOVERNORS OF THE
FEDERAL RESERVE SYSTEM
Washington, D.C., February 16, 2005
THE PRESIDENT OF THE SENATE
THE SPEAKER OF THE HOUSE OF REPRESENTATIVES
The Board of Governors is pleased to submit its Monetary Policy Report to the Congress
pursuant to section 2B of the Federal Reserve Act.
Sincerely,
Alan Greenspan, Chairman
Contents
Page
Monetary Policy and the Economic Outlook 1
Economic and Financial Developments in 2004 and Early 2005 4
Monetary Policy Report to the Congress
Report submitted to the Congress on February 16, 2005, ing and gains in industrial production dropped back nota­
pursuant to section 2B of the Federal Reserve Act bly during the summer. Equity prices and longer-term
interest rates moved lower over this period as well. In the
event, the slowdown in household spending growth proved
MONETARY POLICY AND THE ECONOMIC OUTLOOK
short lived. Both hiring and increases in factory output
The year 2004 was marked by continued expansion in stepped up again in the autumn, and these gains were
economic activity and appreciable gains in employment. extended early this year. With profits healthy and finan­
With fiscal policy stimulative, monetary policy accom­ cial conditions still supportive, capital spending
modative, and financial conditions favorable, household increased at a brisk pace throughout the year. Over the
spending remained buoyant and businesses increased final quarter of 2004, short-term interest rates rose fur­
investment in capital equipment and inventories, despite ther as monetary policy was firmed at each FOMC meet­
the restraint imposed by sizable increases in oil prices. ing, but long-term interest rates were largely unchanged.
Labor market conditions improved significantly, albeit Equity prices rose appreciably in the fourth quarter, and
at an uneven pace, and productivity rose notably further. the dollar depreciated against most other major curren­
Consumer price inflation moved higher with the surge in cies. The FOMC increased the target federal funds rate
energy prices, but core consumer price inflation (that is, 25 basis points again at its meeting this month, bringing
excluding food and energy) remained well contained, and the cumulative tightening over the past year to 11/
2
per­
measures of expected inflation over longer horizons held centage points.
steady or edged lower. The fundamental factors underlying the continued
Although economic activity had increased substantially strength of the economy last year should carry forward
in 2003, the expansion nevertheless appeared somewhat into 2005 and 2006, promoting both healthy expansion
tentative as 2004 opened, in large measure because busi­ of activity and low inflation. Monetary policy is still
nesses still seemed to be reluctant to boost hiring. Over accommodative, and financial conditions more generally
the course of the spring, however, it became clearer that continue to be advantageous for households and firms.
the expansion was solidifying. Businesses added appre­ Profits have been rising briskly, and corporate borrowing
ciably to their payrolls, boosted investment in equipment costs are low. Household net worth has increased with
and software, and started restocking inventories. While the continued sharp rise in the value of real estate assets
household spending growth softened somewhat, residen­ as well as gains in equity prices, and this will likely help
tial construction expanded rapidly. Rising energy prices support consumer demand in the future. Absent a signifi­
boosted overall consumer price inflation, and core infla­ cant increase in oil prices from current levels, the drag
tion moved up as well. In response to positive economic from last year’s run-up should wane this year. The lagged
news and higher inflation during this period, market par­ effects of the decline in the exchange value of the dollar
ticipants came to anticipate that monetary policy tighten­ since the autumn and sustained foreign economic growth
ing would begin sooner than they had expected, and are likely to boost the demand for U.S. exports. The pros­
interest rates increased considerably. With the economic pects for the expansion of aggregate supply also appear
expansion more firmly established and slack in labor and to be quite favorable. Gains in structural labor productiv­
product markets somewhat diminished, the Federal Open ity should continue, although not necessarily at the pace
Market Committee (FOMC) at its June meeting began to of recent years. Economic growth will likely be sufficient
reduce the substantial degree of monetary accommoda­ to generate notable increases in employment, although
tion that was in place. any reversal of the decline in labor force participation
The gradual removal of monetary policy stimulus con­ observed since 2001 would tend to hold up the unem­
tinued in the second half of the year as the economy ployment rate. Core consumer price inflation has remained
expanded at a healthy clip on balance. Around midyear, low since the larger increases posted in the early months
some measures of growth in activity softened, partly of 2004, and long-term inflation expectations have been
because of the drain on income and the rise in business similarly well contained. With some slack likely remain­
costs created by higher oil prices. The expansion of con­ ing in labor and product markets at present and with the
sumer spending slowed in the spring, and the pace of hir­ indirect effects of higher oil and import prices diminish-
2 Monetary Policy Report to the Congress February 2005
ing, the prospects for inflation staying low are good. A By the time of the May and June FOMC meetings,
favorable economic outcome is, of course, not assured, incoming economic data pointed to a broader and more
but at the most recent FOMC meeting the Committee firmly established expansion, with continued strength in
again assessed the risks to both output and inflation as housing markets and business fixed investment. Also, the
balanced. The Committee also reaffirmed that it is pre- employment reports for March, April, and May had indi­
pared to respond to events as necessary in its pursuit of cated strong and widespread gains in private nonfarm
price stability. payrolls, and previous reports for January and February
were revised upward significantly. Overall consumer price
inflation in the first quarter was faster than it had been a
year earlier, and core inflation also increased, in part
Monetary Policy, Financial Markets, and the
because of the indirect effects of higher energy prices.
Economy in 2004 and Early 2005
The Committee maintained its target for the federal funds
In early 2004, against the backdrop of stimulative fiscal rate at 1 percent in May, but on the basis of the evolving
and monetary policy, continued rapid growth in produc­ outlook for economic activity and prices, it revised its
tivity, and supportive financial market conditions, busi­ assessment of risks to indicate that the upside and down-
ness outlays appeared to be firming significantly and side risks for inflation had moved into balance. The Com­
household spending remained strong. The FOMC became mittee also stated that monetary policy accommodation
more confident that the economic expansion was likely could “be removed at a pace that is likely to be mea­
gaining traction and that the risk of significant further sured” to communicate its belief, given its economic out-
disinflation had been greatly reduced. In these circum­ look, that policy would probably soon need to move
stances, it recognized that a highly accommodative stance toward a more neutral stance, though probably not at a
for monetary policy could not be maintained indefinitely. rapid pace. The Committee retained this language at the
Nonetheless, the Committee was concerned about the June meeting while raising its target for the federal funds
persistently slow pace of hiring and viewed underlying rate from 1 percent to 11/
4
percent and noting that it would
inflation pressures as likely to remain subdued. Accord­ “respond to changes in economic prospects as needed to
ingly, the Committee left its target for the federal funds fulfill its obligation to maintain price stability.”
rate unchanged at 1 percent at its January and March The information that the Committee had received by
meetings. However, beginning in January, it modified the the time of its August meeting indicated that economic
language of its policy statement to gain greater flexibil­ growth had softened somewhat earlier in the summer.
ity to tighten policy should circumstances warrant by Although the housing market had remained strong and
indicating that monetary policy accommodation would business outlays had continued to be healthy, consumer
eventually have to be removed. At the same time, the spending growth had slowed significantly, and industrial
Committee suggested that it could be patient in under- production had begun to level off. Also, the June and July
taking such actions. labor market reports revealed that employment growth
Selected interest rates
Percent
Ten-year Treasury 6
5
Two-year Treasury
4
3
Intended federal funds rate 2
1
1/30 3/19 5/7 6/26 8/13 9/24 11/6 12/10 1/29 3/18 5/6 6/25 8/12 9/16 10/28 12/9 1/28 3/16 5/4 6/30 8/10 9/21 11/10 12/14 2/2
2002 2003 2004 2005
NOTE. The data are daily and extend through February 9, 2005. Treasury rates are constant-maturity yields based on the most actively traded securities. The
dates on the horizontal axis are those of FOMC meetings.
SOURCE. Department of the Treasury and the Federal Reserve.
Board of Governors of the Federal Reserve System 3
had slowed considerably. At the same time, core consumer and the news on job growth was uneven—lackluster
price inflation had moderated in May and June even increases in nonfarm payrolls in September were followed
though sizable increases in food and energy prices con­ by robust expansion in October. Inflation measures were
tinued. However, the Committee believed that the soft­ moderate, although up somewhat from one year earlier.
ness in economic activity was caused importantly by On balance, the Committee saw the economy as growing
higher prices of imported oil and would prove short lived. at a pace that would reduce margins of slack in the utili­
With financial conditions remaining stimulative, the zation of resources. The Committee also judged that
economy appeared poised to grow at a pace sufficient to inflationary pressures would likely be well contained if
trim slack in resource utilization. In that regard, given monetary policy accommodation were gradually with-
the unusually low level of the federal funds rate, espe­ drawn. The Committee’s decision to raise its target for
cially relative to the level of inflation, policymakers noted the federal funds rate from 13/
4
percent to 2 percent with
that significant cumulative policy tightening would likely minimal change in the language in the accompanying
be needed to meet the Federal Reserve’s long-run objec­ statement was largely anticipated by financial markets
tives of price stability and sustainable economic growth. and elicited little reaction.
The Committee’s decision at the meeting to raise its At its December meeting, the Committee viewed avail-
target for the federal funds rate 25 basis points, to able information as continuing to indicate that the pace
11/
2
percent, and to maintain its assessment of balanced of the economic expansion was sufficient to further
risks with respect to sustainable growth and price stabil­ reduce the underutilization of resources, despite elevated
ity was largely anticipated by financial markets. How- oil prices. Consumer spending remained solid, investment
ever, market participants revised up their expectations spending was strong, and manufacturing production
for the path of the federal funds rate, reportedly because showed modest growth. Also, employment gains in
the announcement conveyed a somewhat more optimis­ October and November were consistent with gradual
tic outlook for the economy than many had anticipated. improvement in the labor market. Meanwhile, core infla­
By the time of the September FOMC meeting, avail- tion, while above the unusually low rates of late 2003,
able information suggested that the economy had regained remained subdued. Accordingly, the Committee voted to
momentum. Real consumer spending bounced back raise its target for the federal funds rate 25 basis points,
sharply in July after a weak second quarter, and incom­ to 21/
4
percent, and to retain the previous statement that
ing data on industrial production indicated a modest the removal of policy accommodation would likely be
strengthening. Housing activity had increased further, and “measured.” Investors had largely anticipated the policy
business outlays had picked up significantly in the sec­ rate decision, but a few market participants had report­
ond quarter. In addition, the labor market showed signs edly speculated that the Committee would signal increased
of improvement in August, as the unemployment rate concern about inflationary pressures. In the absence of
edged down and nonfarm payrolls grew moderately. Core any such signal, implied rates on near-dated futures con-
consumer price inflation slowed in June and July, and a tracts and longer-term Treasury yields declined a few basis
decline in energy prices from record levels pushed down points after the release of the December statement.
readings on headline inflation. Although the Committee Also at its December meeting, the Committee consid­
acknowledged that higher oil prices had damped the pace ered an accelerated release of the minutes of FOMC meet­
of economic activity around midyear, it nonetheless saw ings. The Committee’s practice had been to publish the
the expansion as still on solid footing. Consequently, the minutes for each meeting on the Thursday after the next
Committee agreed to increase its target for the federal scheduled meeting. The Committee believed that, because
funds rate another 25 basis points, to 13/
4
percent; to reit­ the minutes contain a more nuanced explanation of policy
erate its view that the risks to price stability and to sus­ decisions than the statement released immediately after
tainable growth were balanced; and to repeat its indica­ each meeting, publishing them on a timelier basis would
tion that the removal of policy accommodation would help market participants interpret economic developments
likely proceed at a “measured” pace. The reaction in and thereby better anticipate the course of interest rates.
financial markets to the policy rate decision and the Earlier release would also provide a context for the pub­
accompanying statement was muted. lic remarks of individual FOMC members. It was also
The information in hand at the time of the November recognized, however, that financial markets might misin­
FOMC meeting generally suggested that the economy had terpret the minutes at times and that earlier release might
continued to expand at a moderate rate despite the adversely affect the Committee’s discussions and, per-
restraint that higher oil prices imparted to real incomes haps, the minutes themselves. After weighing these con­
and consumer confidence. Consumer and business spend­ siderations, the Committee voted unanimously to pub­
ing stayed firm, and the housing market remained buoy- lish the FOMC minutes three weeks after the day of the
ant. However, industrial production was about unchanged, policy decision.
4 Monetary Policy Report to the Congress February 2005
The information that the Committee reviewed at its the chain-type price index for personal consumption
February 2005 meeting indicated that the economy had expenditures excluding food and energy (core PCE) will
continued to expand at a steady pace. The labor market increase between 11/
2
percent and 13/
4
percent both this
showed signs of further improvement, and consumer year and next—about the same as the 1.6 percent increase
spending and the housing market remained robust. posted over 2004.
Industrial production accelerated, particularly at the end
of 2004, and growth of business fixed investment was
solid in the fourth quarter. Core inflation stayed moder­ ECONOMIC AND FINANCIAL DEVELOPMENTS
ate, and measures of inflation expectations remained well IN 2004 AND EARLY 2005
anchored. Given the solid economic expansion and lim­
The economy proved to be sufficiently resilient to main­
ited price pressures, the Committee voted to continue its
tain solid growth and moderate core inflation in 2004
removal of policy accommodation by raising its target
even as higher oil prices drained consumers’ purchasing
f
a
o
n
r
d
t h
to
e f
e
e
s
d
se
e
n
ra
ti
l
a
f
l
u
ly
n d
r
s
e p
ra
e
t
a
e
t
f
t
r
h
o
e
m
l a
2
n
1
g
/ 4
u
p
a
e
g
r
e
c e
o
n
f
t
t
t
h
o
e
2
D
1/ 2
e c
p
e
e
m
rc
b
en
er
t power and boosted firms’ costs. Real GDP rose 33/
4
per-
statement. Futures market quotes indicated that investors
cent last year after having increased 41/
2
percent in 2003.
Activity was supported by continued robust advances in
had already priced in a 25 basis point increase in the tar-
household spending. In addition, capital spending by
get federal funds rate at the meeting, and market partici­
businesses increased notably. Labor market conditions
pants reportedly expected no substantive changes to the
improved significantly, though at an uneven pace over
accompanying statement. Accordingly, the reaction in fi­
the course of the year. Private payrolls, which turned up
nancial markets to the announcement was minimal.
in late 2003, rose 170,000 per month last year, on aver-
age, and the unemployment rate declined below 51/
2
per-
Economic Projections for 2005 and 2006 cent by year-end and to 51/
4
percent in January 2005—
the lowest rates since 2001.
Federal Reserve policymakers expect the economy to
Consumer price inflation was driven higher last year
expand moderately and inflation to remain low in 2005
by the sharp rise in energy prices. Although core con­
and 2006.1 The central tendency of the forecasts of real
sumer price inflation moved up somewhat from unusu­
GDP growth made by the members of the Board of Gov­
ally low levels recorded in 2003, it remained well con­
ernors and the Federal Reserve Bank presidents is
tained. Price increases were restrained by continuing,
33/
4
percent to 4 percent over the four quarters of 2005.
though diminishing, slack in labor and product markets,
The civilian unemployment rate is expected to average
which tended to offset the effects of higher energy and
about 51/
4
percent in the fourth quarter of 2005. For 2006,
commodity prices, as well as the weaker dollar, on firms’
the policymakers project real GDP to increase about
overall costs. In addition, solid productivity gains
31/
2
percent, and they expect the unemployment rate to
implied that unit labor costs rose only modestly, even if
edge down to between 5 percent and 51/
4
percent. With
up from the declines in the preceding two years. The
regard to inflation, FOMC participants project that
decline in crude oil prices, on balance, since October
points to some easing of cost pressures on firms from
1. As a further step to enhance monetary policy communications, that source in the period ahead.
Federal Reserve policymakers will now provide economic projections
Several forces likely contributed to last year’s impres­
for two years, rather than one, in the February Monetary Policy
Report. sive economic performance in the face of the sizable
Economic projections for 2005 and 2006
Percent
Federal Reserve Governors and Reserve Bank presidents
Indicator 200 M 4 E a M c O tu al 2005 2006
Range Central Range Central
tendency tendency
Change, fourth quarter to fourth quarter1
Nominal GDP ........................................................................ 6.2 5–6 5½–5¾ 5–5¾ 5 –5½
Real GDP .............................................................................. 3.7 3½–4 3¾–4 3¼–3¾ 3½
PCE price index excluding food and energy ........................ 1.6 1½–2 1½–1¾ 1½–2 1½–1¾
Average level, fourth quarter
Civilian unemployment rate .................................................. 5.4 5–5½ 5¼ 5–5¼ 5 –5¼
1. Change from average for fourth quarter of previous year to average for fourth quarter of year indicated.
Board of Governors of the Federal Reserve System 5
Change in real GDP spreads on corporate bonds narrowed, and commercial
banks eased terms and standards on business loans. In
Percent, annual rate this environment, household debt again increased briskly.
The borrowing needs of nonfinancial businesses were
damped by their strong cash flows. Equity values rose,
6 especially toward the end of the year. At the same time,
the exchange value of the dollar declined, on net, over
the year as market participants apparently focused on the
4 financing implications of the large and growing U.S. cur-
rent account deficit.
2
The Household Sector
Consumer Spending
1998 2000 2002 2004
NOTE. Here and in subsequent charts, except as noted, change for a given Consumer spending grew substantially last year. Personal
period is measured to its final quarter from the final quarter of the preceding consumption expenditures (PCE) advanced nearly 4 per-
period.
SOURCE. Department of Commerce, Bureau of Economic Analysis. cent in real terms, about the same as the increase in 2003.
Sales of new motor vehicles remained brisk, on average,
adverse oil shock. The growth of real output continued at 163/
4
million units. Excluding motor vehicles, consumer
to be undergirded by gains in structural labor productiv­ spending on most categories of durable and nondurable
ity. Moreover, fiscal policy remained stimulative last year goods rose rapidly, as gains in real expenditures for food
through the combination of the lagged effect of earlier and clothing both exceeded 5 percent; however, spend­
cuts in personal tax rates, the rise in defense spending, ing on computing equipment increased less in 2004 than
and perhaps also the partial-expensing tax incentives for in preceding years, and consumers responded to the high
business investment. Monetary policy was highly accom­ cost of gasoline and heating fuel by cutting back on real
modative in the early part of the year and remained spending for these items. Real outlays for services also
accommodative, though progressively less so through- increased rapidly last year, and medical services posted
out the year, and credit remained readily available at especially large gains.
favorable terms. Consumer demand was also boosted by Real disposable personal income (DPI) rose nearly
the strong increases in asset values during the past two 4 percent last year, but this figure is exaggerated by
years. Microsoft’s $32 billion special dividend payment in
Financial conditions remained stimulative last year December (the bulk of which is estimated to have
even as market participants revised up their expectations accrued to U.S. households). If this one-time event
for the near-term path of monetary policy. Interest rates is excluded from the calculation, real DPI rose only
on longer-term Treasury securities remained low, risk 23/
4
percent in 2004, well below the increase posted in
2003. Faster job growth helped to support increases in
Change in PCE chain-type price index
Change in real income and consumption
Percent
Percent, annual rate
Total
Excluding food and energy Disposable personal income
Personal consumption expenditures
3
6
2
4
1
2
1998 2000 2002 2004
1998 2000 2002 2004
NOTE. The data are for personal consumption expenditures (PCE).
SOURCE. Department of Commerce, Bureau of Economic Analysis. SOURCE. Department of Commerce, Bureau of Economic Analysis.
6 Monetary Policy Report to the Congress February 2005
households’ incomes last year in nominal terms, and the Wealth-to-income ratio
Jobs and Growth Tax Relief Reconciliation Act of 2003
(JGTRRA), which brought lower personal tax rates for- Ratio
ward into 2003, led to larger refunds and smaller final
payments in the spring of 2004. However, real income
gains were held down, as higher oil prices siphoned off 6
household purchasing power.
With the growth of real consumption spending out-
pacing that of real income through most of last year, the 5
personal saving rate moved lower, from 11/
2
percent, on
average, in 2003 to only 1/
2
percent in the third quarter of
last year. (The fourth-quarter surge in income associated 4
with the Microsoft dividend payments pushed the saving
rate back up to 11/
4
percent, but this increase will likely
be reversed early this year as dividend income falls back. 1984 1988 1992 1996 2000 2004
Because the company’s share price declined in step with
NOTE. The data are quarterly and extend through 2004:Q3. The wealth-
the dividend payouts, the dividends had no effect on share- to-income ratio is the ratio of household net worth to disposable personal
income.
holders’ overall financial resources and so probably had SOURCE. For net worth, Federal Reserve Board, flow of funds data; for
little effect on consumption.) income, Department of Commerce, Bureau of Economic Analysis.
Low interest rates were one factor that helped to sup-
port consumption growth—especially for durable wealth feeds through into household spending over a
goods—despite comparatively slow gains in real income. period of several quarters, the wealth increases in both
Higher household wealth was also an important force that 2003 and 2004 were important in supporting consumer
propelled consumer spending last year. According to the spending last year. The rise in house prices, together with
Federal Reserve’s flow of funds accounts, the ratio of continued low interest rates, also led consumers to
household net worth to disposable income rose sharply extract additional equity from their homes, in particular
in 2003, as corporate equity values rebounded and home through home equity loans. Such actions provided many
prices continued to rise. Moreover, although equity val­ households with a readily available and relatively low-
ues were little changed, on net, through much of 2004 cost source of funds for financing consumption.
before rising notably in the final quarter, home prices Consumer confidence, which had improved in 2003,
continued to rise throughout the year, and the wealth-to- remained at generally favorable levels last year, accord­
income ratio moved up further; by the third quarter (the ing to surveys by both the Michigan Survey Research
most recent period for which the complete wealth data Center (SRC) and the Conference Board. Confidence
are available), the ratio had reversed nearly half its tended to dip at times during the year when energy prices
decline since the stock market peak in 2000. Because were moving up most rapidly, but it recovered soon after
those episodes.
Consumer sentiment
Personal saving rate
1985 = 100 1966 = 100
Percent
140 140
12
Conference Board
120 120
9
100 100
Michigan SRC
6 80 80
3 60 60
1993 1996 1999 2002 2005
1984 1988 1992 1996 2000 2004
NOTE. The data are monthly and extend through January 2005.
NOTE. The data are quarterly and extend through 2004:Q4. SOURCE. The Conference Board and University of Michigan Survey
SOURCE. Department of Commerce, Bureau of Economic Analysis. Research Center.
Board of Governors of the Federal Reserve System 7
Mortgage rates Change in house prices
Percent Percent
Fixed rate 12
7
Repeat-transactions index 9
5 6
Adjustable rate
3
3 New home price
index +
0_
2001 2002 2003 2004 2005 1984 1988 1992 1996 2000 2004
NOTE. The data, which are weekly and extend through February 9, 2005, NOTE. The repeat-transactions index includes purchase transactions only
are contract rates on thirty-year mortgages. and extends through 2004:Q3. The new home price index extends through
SOURCE. Federal Home Loan Mortgage Corporation. 2004:Q4. Change is over four quarters.
SOURCE. For repeat transactions, Office of Federal Housing Enterprise
Oversight; for new home prices, Department of Commerce, Bureau of the
Census.
Residential Investment
Residential investment remained robust last year. Real
rapid pace of 1.5 million units started in 2003. In the
expenditures increased 53/
4
percent in 2004—the third
multifamily sector, starts totaled a solid 350,000 units
straight year of strong gains. Demand for housing was
last year, a figure in line with that of the preceding sev­
influenced by the same factors that affected household
eral years. Sales of both new and existing single-family
spending more generally, but it was especially supported
homes hit new highs last year, and home prices moved
by nominal mortgage interest rates that have remained
up sharply. The repeat-transactions price index for exist­
near their lowest levels since the late 1960s. Rates on
ing homes (limited to purchase transactions only), which
thirty-year fixed-rate mortgages fluctuated between about
is published by the Office of Federal Housing Enterprise
51/
2
percent and 61/
4
percent over the past two years; they
Oversight, climbed more than 10 percent over the four
edged up to the high end of that range during the spring
quarters ending in the third quarter of last year (the latest
but dropped back to under 6 percent by the end of sum­
quarter for which data are available) and is up a cumula­
mer and now stand below 53/
4
percent.
tive 65 percent since 1997, when it started to rise notably
In the single-family sector, housing starts amounted
more rapidly than overall inflation. These price increases
to 1.6 million units last year, a rate faster than the already
have also outstripped by a wide margin the increases in
household incomes and rents. Another nationwide price
index, the Census Bureau’s constant-quality price index
Private housing starts for new homes, rose only 63/
4
percent last year. Because
this index does not adjust for the location of new homes
Millions of units, annual rate
within metropolitan areas, and because new homes con­
stitute only a small fraction of the overall housing stock,
this index is probably a less reliable indicator of overall
1.6
home values than is the repeat-transactions index.
Single-family
1.2
Household Finance
.8
Household debt is estimated to have increased about
Multifamily 93/
4
percent in 2004, a touch less than in the previous
.4
year. Mortgage debt again paced this advance. The brisk
expansion of mortgages reflected continued strong
activity in housing markets and rising house prices. How-
1992 1994 1996 1998 2000 2002 2004
ever, the growth rate of mortgage debt did not quite match
NOTE. The data are quarterly and extend through 2004:Q4.
SOURCE. Department of Commerce, Bureau of the Census. that registered in 2003. Refinancing activity fell off
8 Monetary Policy Report to the Congress February 2005
Household financial obligations ratio hold financial obligations and debt service, which cap­
ture pre-committed expenditures relative to disposable
Percent income, were little changed last year, on balance, though
they remained high by historical standards. Nevertheless,
20 measures of household credit quality either held steady
or improved during the course of the year. The latest avail-
19
able data indicate that delinquency rates on credit card
18 loans, consumer loans, and residential mortgages at
commercial banks declined, while those on auto loans at
17 captive finance companies were about unchanged at a
low level. Household bankruptcy filings ran below the
16
elevated levels of 2003, although they stayed generally
15 above the rates posted in earlier years.
1992 1995 1998 2001 2004
The Business Sector
NOTE. The data are quarterly and extend through 2004:Q4. The final
observation, 2004:Q4, is a projection. The financial obligations ratio equals
the sum of required payments on mortgage and consumer debt, automobile Fixed Investment
leases, rent on tenant-occupied property, homeowners’ insurance, and
property taxes, all divided by disposable personal income.
Business fixed investment rose robustly for a second con­
secutive year in 2004. Real spending on equipment and
sharply last year, as the pool of outstanding mortgages software (E&S) increased 131/
2
percent, about as much
with interest rates above current market rates shrank con­ as in 2003, as firms’ final sales continued to increase,
siderably. Mortgages with adjustable interest rates, profits and cash flow rose further, and many businesses
including hybrids that feature both fixed and adjustable
interest rate components, were increasingly popular in
2004. Consumer credit continued to expand at a moder­ Change in real business fixed investment
ate pace by historical standards, restrained in part by the
Percent, annual rate
substitution of other forms of debt, such as home equity
Structures
loans. Higher interest rates on some consumer loans and
Equipment and software
credit cards in the second half of 2004 may have also 20
damped the growth of consumer credit.
Relatively low interest rates and further gains in dis­ 10
posable personal income limited pressures on household +
balance sheets in 2004. Measures of aggregate house-
0_
10
Delinquency rates on selected types of household loans
20
Percent
High-tech equipment and software
Credit card pools 6
Other equipment excluding transportation
30
5
20
4
Auto loans at domestic auto finance companies
3 10
+
Mortgages
2 0_
1
10
1992 1994 1996 1998 2000 2002 2004
1998 2000 2002 2004
NOTE. The data are quarterly. The rates for credit card pools and mortgages
extend through 2004:Q3; the rate for auto loans extends through 2004:Q4. NOTE. High-tech equipment consists of computers and peripheral equip­
SOURCE. For credit cards, Moody’s Investors Service; for auto loans, Big ment, software, and communications equipment.
Three automakers; for mortgages, Mortgage Bankers Association. SOURCE. Department of Commerce, Bureau of Economic Analysis.
Board of Governors of the Federal Reserve System 9
reported a need to replace or upgrade existing equipment Change in real business inventories
and software. Although many firms had little need to
seek outside financing given their flush cash situation, Billions of chained (2000) dollars, annual rate
those that did generally found financial markets to be
receptive—interest rates remained low and other terms 75
and conditions stayed relatively favorable. The partial-
expensing tax incentives, which covered new equipment 50
and software installed by the end of 2004, boosted prof-
its and cash flow and may have also stimulated some 25
investment spending.
+
Increases in E&S spending were fairly widespread 0_
across categories of capital goods. Spending on high-
25
technology equipment increased 15½ percent last year
after having risen 19 percent in 2003; these gains fol­
lowed two years of declines. Although the pattern of 1998 2000 2002 2004
spending was uneven over the four quarters of 2004, for
SOURCE. Department of Commerce, Bureau of Economic Analysis.
the year as a whole, business outlays for computing equip­
ment rose 25 percent in real terms, while spending on
software and communications equipment posted increases
of 13 percent and 10 percent respectively. Outside of the
tories that were increasingly lean relative to sales. In 2004,
high-tech sector, business spending on aircraft moved
when stocks had become quite spare—even after taking
lower for the third consecutive year, as airlines contin­
into account the ongoing improvements in inventory
ued to struggle with a highly competitive market envi­
management that have allowed firms to economize on
ronment and high fuel prices. In contrast, business out-
stockholding—and businesses had apparently grown more
lays on motor vehicles rose substantially last year, with
confident in the durability of the recovery, businesses
the demand for trucks exceptionally strong. Investment
accumulated $45 billion of inventories (in real terms),
in equipment other than high-tech and transportation
according to preliminary data. The step-up in the pace of
goods—a category that includes industrial machinery
stockbuilding contributed about 1/
4
percentage point to
and a wide range of other types of equipment—moved
GDP growth last year.
up 11 percent last year, the most in more than ten years.
In contrast to the rebound in equipment spending, real
outlays in the nonresidential construction sector were
Corporate Profits and Business Finance
about unchanged for a second year in 2004 and have yet
to recover from their sharp downturn during 2001 and Strong growth of corporate profits again allowed many
2002. In the office sector, where construction increased firms to finance capital spending with internal funds last
rapidly in the late 1990s, spending has remained espe­ year. As a result, nonfinancial business debt rose at only
cially weak; vacancy rates for these properties, although a moderate pace. Net equity issuance dropped further into
down a touch over the past year, are still quite elevated. negative territory in 2004, and on balance nonfinancial
Construction of industrial buildings has also remained corporations are estimated to have raised no net funds in
low as a result of high vacancy rates. In contrast, demand credit and equity markets. However, short-term business
for new retail and wholesale properties has been firmer, debt, including commercial paper and commercial and
reportedly a reflection of the steady increases in consumer industrial (C&I) loans, expanded last year after three years
spending, and outlays for these types of buildings moved of contraction, and commercial mortgage debt continued
higher last year. In addition, investment in the drilling to increase rapidly. The credit quality of businesses
and mining sector rose last year in response to high prices remained strong.
for natural gas. Corporate profits held up well in 2004 after surging in
the previous year. The ratio of before-tax profits of non-
financial corporations to that sector’s gross value added
increased for a second consecutive year. In the fourth
Inventory Investment
quarter of 2004, operating earnings per share for S&P
Businesses added appreciably to inventories last year for 500 firms were nearly 20 percent above their level four
the first time since running down their holdings sharply quarters earlier. Analysts’ earnings forecasts began to
in 2001. As economic activity strengthened during 2002 moderate somewhat in the second half of 2004 after sev­
and 2003, many businesses chose to operate with inven- eral months of strong upward revisions.
10 Monetary Policy Report to the Congress February 2005
Before-tax profits of nonfinancial corporations Selected components of net business financing
as a percent of sector GDP
Billions of dollars
Percent
Commercial paper
Bonds
14 Bank loans Sum of selected 400
components
12 200
+
10
0_
8
200
6
2002 2003 2004
1980 1984 1988 1992 1996 2000 2004
NOTE. Seasonally adjusted annual rate for nonfinancial corporate business.
NOTE. The data are quarterly and extend through 2004:Q3. Profits are from The data for the sum of selected components are quarterly. The data for
domestic operations of nonfinancial corporations, with inventory valuation 2004:Q4 are estimated.
and capital consumption adjustments. SOURCE. Federal Financial Institutions Examination Council, Consolidated
SOURCE. Department of Commerce, Bureau of Economic Analysis. Reports of Condition and Income (Call Report).
In equity markets, net issuance of shares by nonfinan­ issue bonds. Firms reportedly used a large portion of the
cial firms turned more negative in 2004. Although initial proceeds to pay down existing debt, although some com­
public offerings rebounded from the sluggish pace of panies used the funds raised in the bond market to repur­
the past two years, ample profits and sizable cash hold­ chase equity shares or to finance mergers.
ings helped boost share retirements from mergers and Short-term business borrowing revived in 2004 after
repurchases. a prolonged contraction. Commercial paper outstanding
Net corporate bond issuance was sluggish in 2004, as turned up in the first half of the year, although it flattened
firms evidently relied heavily on their considerable prof- out over the second half. Business loans at banks
its to fund investment in fixed capital and inventories. rebounded over the course of last year. According to
The timing of gross bond issuance was influenced by results from the Federal Reserve’s Senior Loan Officer
interest rate movements during the year, as firms took
advantage of occasional dips in longer-term yields to
Net percentage of domestic banks tightening
standards on commercial and industrial loans
Financing gap and net equity retirement to large and medium-sized firms
at nonfinancial corporations
Percent
Billions of dollars
300 60
250
Net equity retirement 40
200
150 20
100 +
Financing gap
0_
50
+
0_ 20
50
1991 1993 1995 1997 1999 2001 2003 2005
1990 1992 1994 1996 1998 2000 2002 2004
NOTE. The data are based on a survey generally conducted four times per
NOTE. The data are annual; 2004 is based on partially estimated data. The year; the last reading is from the January 2005 survey. Large and
financing gap is the difference between capital expenditures and internally medium-sized firms are those with annual sales of $50 million or more. Net
generated funds. Net equity retirement is the difference between equity percentage is the percentage reporting a tightening less the percentage
retired through share repurchases, domestic cash-financed mergers, or foreign reporting an easing.
takeovers of U.S. firms and equity issued in public or private markets, SOURCE. Federal Reserve, Senior Loan Officer Opinion Survey on Bank
including funds invested by venture capital partnerships. Lending Practices.
Board of Governors of the Federal Reserve System 11
Opinion Survey on Bank Lending Practices, commercial Default rate on outstanding corporate bonds
banks eased terms and standards on business loans dur­
ing the course of 2004 in response to the improved eco­ Percent
nomic outlook and to increased competition from other
banks and nonbank lenders. Survey responses also indi­ 4
cated an increase in demand for C&I loans that reflected
firms’ need to fund rising accounts receivable, invento­ 3
ries, capital expenditures, and merger activity. Concerns
over loan quality seemed to diminish further in 2004, as 2
spreads on leveraged deals in the syndicated loan market
edged down from already low levels. 1
Corporate credit quality remained solid in 2004 amid +
strong earnings, low interest rates, and a further buildup
0_
of already substantial cash positions on firms’ balance
sheets. The delinquency rate on C&I loans declined fur­ 1992 1994 1996 1998 2000 2002 2004
ther, and the twelve-month trailing default rate on corpo­ NOTE. The data are monthly and extend through December 2004. The rate
rate bonds fell to historically low levels before edging up for a given month is the face value of bonds that defaulted in the twelve
months ending in that month divided by the face value of all bonds
late in the year. Net upgrades of bonds by Moody’s outstanding at the end of the calendar quarter immediately preceding the
Investor Service for both investment- and speculative- twelve-month period.
SOURCE. Moody’s Investors Service.
grade nonfinancial firms increased last year.
The stock of commercial mortgage debt outstanding
The Government Sector
grew at a rapid pace in 2004. Some firms reportedly con­
tinued to find mortgages an attractive source of long-term Federal Government
funding. The expansion of commercial mortgage credit
helped propel issuance of commercial-mortgage-backed The federal budget position deteriorated slightly further
securities (CMBS) to near-record levels. Delinquency in 2004, as spending increases and further tax reductions
offset the effects of stronger economic growth on rev­
rates on commercial mortgages on the books of banks
enues. The unified budget deficit widened from $378 bil­
and insurance companies remained low throughout the
lion in fiscal 2003 to $412 billion in fiscal 2004. As a
year, and those on loans backing mortgage securities fell.
share of GDP, the federal unified deficit stood close to
Considerable gains in commercial real estate prices
increased owners’ equity and largely kept pace with the
31/
2
percent in both years. Receipts increased 51/
2
percent
in fiscal 2004 after two years of declines. Corporate
sizable increase in mortgage debt obligations. Yield
receipts surged more than 40 percent, or $58 billion,
spreads of CMBS over comparable Treasury securities
reflecting the improvement in corporate profits; individual
remained moderate.
tax receipts—restrained by JGTRRA, which pulled for-
Net interest payments of nonfinancial corporations
Federal receipts and expenditures
as a percent of cash flow
Percent of nominal GDP
Percent
24
Expenditures
20
22
Receipts
Expenditures 20
15 excluding net interest
18
10
16
1986 1989 1992 1995 1998 2001 2004
1980 1984 1988 1992 1996 2000 2004
NOTE. The budget data are from the unified budget and are for fiscal years
NOTE. The data are quarterly and extend through 2004:Q3. (October through September); GDP is for the year ending in Q3.
SOURCE. Department of Commerce, Bureau of Economic Analysis. SOURCE. Office of Management and Budget.
12 Monetary Policy Report to the Congress February 2005
ward reductions of personal tax rates that had been sched­ in the autumn provided disaster aid for victims of hurri­
uled for the second half of the decade—rose only about canes and for ranchers and farmers affected by drought
2 percent. Overall federal receipts increased less rapidly conditions.
than nominal GDP, and the ratio of receipts to GDP edged The recent sizable deficits in the unified budget mean
down to 161/
4
percent, the lowest level in more than forty that the federal government, which had been contribut­
years. ing to the pool of national saving from 1997 through 2000,
Meanwhile, nominal federal outlays increased about has been drawing on that pool since 2001. Net federal
6 percent in fiscal 2004. Spending for national defense saving—essentially the unified budget balance adjusted
increased especially sharply, but spending also increased to the accounting practices of the national income and
notably for Medicare and Medicaid. Debt service costs, product accounts (NIPA)—dropped from positive 2 per-
which fell sharply from 1997 through 2003 as a result of cent of GDP in 2000 to a level below negative 3 percent
reduced debt and declining interest rates, edged higher of GDP in 2003 and 2004. Personal saving moved lower
last year. Federal government purchases of goods and ser­ over this period as well, while business net saving rose
vices—the part of spending that is counted in GDP— with the rebound in corporate profits. In all, net national
rose about 4 percent in real terms in 2004 after larger saving edged up in 2004 but remained near its postwar
increases in the preceding two years. (Government spend­ lows. Because net national saving has fallen increasingly
ing on items such as interest payments and transfers is short of net domestic investment over the past several
excluded from GDP because these items do not consti­ years, the inflow of foreign funds needed to finance that
tute a direct purchase of final production.) investment has risen. The growing inflow of foreign capi­
Regarding legislative initiatives, two new tax bills were tal is mirrored in the widening of the nation’s current
enacted in the fall of 2004. First, the Working Families account deficit. Over time, the low national saving rate
Tax Relief Act extended through 2010 a variety of per­ could eventually slow the rise in living standards either
sonal tax reductions that had previously been set to by increasing the burden of servicing U.S. foreign debt
expire earlier. Second, the American Jobs Creation Act or by impinging on domestic capital formation.
replaced the exclusion of extraterritorial income (which The growth rate of Treasury debt moderated slightly
the World Trade Organization had declared an illegal last year after increasing substantially in 2003. Nonethe­
export subsidy) with numerous other tax reductions for less, federal debt held by the public as a percentage of
domestic manufacturers and U.S. multinationals. The first GDP continued to edge higher over the course of 2004
bill is expected to have a ten-year budget cost of around and currently stands at about 361/
2
percent. To help
$150 billion, while the second bill was scored as being finance substantial budget deficits, the Treasury issued
revenue neutral. As for federal spending in fiscal 2005, a considerable volume of bills as well as two-, three-,
the regular appropriations bills provided for sizable five-, and ten-year nominal notes. In addition, the Trea­
increases in spending on defense and homeland security sury expanded its borrowing program in 2004 by adding
and for modest increases in nondefense discretionary semiannual auctions of twenty-year inflation-protected
expenditures. In addition, emergency legislation passed bonds and five-year inflation-protected notes.
Net saving
Change in real government expenditures
on consumption and investment Percent of nominal GDP
Percent 12
Federal Nonfederal saving
State and local 9
9
6
6 3
Total
+
0_
3 Federal saving
3
+
0_
1986 1988 1990 1992 1994 1996 1998 2000 2002 2004
NOTE. The data are quarterly and extend through 2004:Q3. Nonfederal
1998 2000 2002 2004 saving is the sum of personal and net business saving and the net saving of
state and local governments.
SOURCE. Department of Commerce, Bureau of Economic Analysis. SOURCE. Department of Commerce, Bureau of Economic Analysis.
Board of Governors of the Federal Reserve System 13
Federal government debt held by the public resorted to accounting devices, suspended issuance of
state and local government series securities, and post­
Percent of nominal GDP poned a four-week bill auction. In mid-November, Con­
gress raised the debt ceiling from $7.4 trillion to
$8.1 trillion, and the Treasury subsequently resumed nor­
55
mal financing operations.
45
State and Local Governments
35
Pressures on the budgets of state and local governments
have eased as economic activity has strengthened. Tax
25 receipts have been spurred by the increases in household
income, consumer spending, and property values. As a
result, many states seem to be on track to meet balanced
1964 1974 1984 1994 2004
budget requirements in the current fiscal year (which ends
NOTE. Through 2003, the data for debt are year-end figures, and the
June 30 for all but a few states) without using as much
corresponding value for GDP is for Q4 at an annual rate; the final observation
is for 2004:Q3. Excludes securities held as investments of federal gov­ borrowing or other extraordinary measures as in recent
ernment accounts.
years. Nevertheless, a number of states still must deal
with lingering fiscal problems, particularly depleted
Various indicators suggested a continued strong
reserve funds, the expiration of temporary tax hikes, and
appetite for Treasury securities among foreign investors
rising Medicaid costs. In addition, several states still face
last year. Indirect bidding at Treasury auctions, which
serious structural imbalances in their budgets.
includes bidding by the Federal Reserve Bank of New
Real expenditures by state and local governments as
York on behalf of foreign official institutions, remained
measured in the NIPAs remained about flat for a second
robust, and Treasury securities held in custody at the Fed­
year in 2004. Real spending on current operations rose
eral Reserve Bank of New York on behalf of such institu­
less than 1 percent last year, while real investment spend­
tions increased just over $200 billion in 2004. Also, data
ing declined. However, even as they were holding the
from the Treasury International Capital System showed a
line on spending increases, states and localities were able
substantial increase in holdings of Treasury securities by
to resume net hiring in 2004 after having left employ­
foreign official and private investors, particularly those
ment about unchanged in 2003.
in Japan. The proportion of Treasury securities held by
Net issuance of debt by state and local governments
foreign investors is estimated to have risen to a record
edged down from the rapid pace set in 2003, as improved
431/
2
percent by the third quarter of 2004.
budget positions permitted some contraction in short-term
Treasury debt reached its statutory ceiling late last year.
debt. Advance refunding offerings were again strong dur-
To cope with the constraint, the Treasury temporarily
State and local government net saving
Treasury securities held by foreign investors
as a share of total outstanding Percent of GDP
Percent
.5
45
+
0_
40
35 .5
30
1983 1986 1989 1992 1995 1998 2001 2004
NOTE. The data, which are quarterly, are on a national income and product
1997 1998 1999 2000 2001 2002 2003 2004 account basis and extend through 2004:Q3. Net saving excludes social
insurance funds.
NOTE. The data are quarterly and extend through 2004:Q3. SOURCE. Department of Commerce, Bureau of Economic Analysis.
14 Monetary Policy Report to the Congress February 2005
ing the year, as states and municipalities took advantage After increasing at an annual rate of almost 6 percent
of low long-term interest rates and moderate credit in the first half of 2004, prices of exported goods moved
spreads. Credit quality of tax-exempt borrowers improved up at just a 21/
2
percent rate in the second half. This
in 2004. Rating upgrades of tax-exempt bonds outpaced deceleration was due in large part to a reversal of the
downgrades, especially later in the year. run-up in the prices of agricultural products that had
occurred in late 2003 and early 2004. Better harvests last
year returned prices of agricultural products to levels near
The External Sector those that had prevailed before the spike.
Solid growth in income in the United States spurred
After narrowing in 2003, the U.S. current account deficit
widened again last year and was $660 billion (annual rate),
growth of real imports of 91/
2
percent in 2004. The
increase primarily reflected higher imports of goods that
or 5.6 percent of GDP, in both the second and third quar­
occurred despite a notable rise in their prices. Real oil
ters. Much of this widening reflected a considerable
imports expanded almost 10 percent in 2004. Imports of
increase in the deficit on goods and services trade, as a
capital equipment increased throughout the year, but
marked rise in imports more than offset solid increases
imports of consumer goods suffered a period of weak­
in exports. The trade deficit expanded from $500 billion
ness through the middle of the year before rebounding
during the fourth quarter of 2003 to more than $650 bil­
in the fourth quarter. Imports of services moved up only
lion, on average, during the second half of 2004.
13/
4
percent in 2004.
Prices of imported non-oil goods increased at an
annual rate of just over 4 percent in the first half of 2004,
International Trade
but the pace slowed to 2 percent in the second half. This
Real exports of goods and services rose an estimated step-down largely reflected a deceleration in the prices
51/
2
percent in 2004 despite a deceleration in the fourth of industrial supplies, driven by a leveling off of nonfuel
quarter. In the first half, exports were supported by the commodity prices at the elevated levels reached in March.
lagged effect of the fall in the dollar’s value in 2003. Declines in the prices of foods offset continued price
Strong expansion of foreign economic activity also helped increases for metals.
boost exports in the first half, but that stimulus dimin­ The spot price of West Texas intermediate (WTI) crude
ished in the second half of the year when foreign growth oil moved up during most of 2004 and surged tempo­
slowed. For the year as a whole, exports of industrial sup- rarily to a record high of $55 per barrel in October. Since
plies and capital goods posted solid growth. Exports to then, it has fluctuated somewhat below that peak but still
Canada, Mexico, and western Europe rose smartly in at levels well above $33 per barrel, the price at which it
2004, whereas exports to Japan were relatively weak. Real started 2004. Oil prices were driven up by intensified
exports of services increased about 31/
2
percent through concerns that oil supply would not keep pace with sur­
2004 as a whole. prisingly strong global demand. Oil consumption in China
grew nearly 15 percent in 2004, pushing that economy
past Japan as the world’s second-largest consumer. As
oil prices rose, OPEC increased its oil production,
U.S. trade and current account balances
Percent of nominal GDP Change in real imports and exports of goods and services
+
Percent
0_
Imports
1 Exports 15
2
Trade 10
3
Current account
5
4
+
5
0_
6 5
10
1997 1998 1999 2000 2001 2002 2003 2004
NOTE. The data are quarterly. The trade data extend through 2004:Q4 and
1998 2000 2002 2004
the current account data extend through 2004:Q3.
SOURCE. Department of Commerce. SOURCE. Department of Commerce and Federal Reserve staff estimates.
Board of Governors of the Federal Reserve System 15
Prices of oil and of nonfuel commodities down some production. Russian oil output, however, con­
tinued despite the breakup of Yukos, formerly Russia’s
January 2001 = 100 Dollars per barrel largest oil company. Late in the year, oil prices declined
from their October highs, as production recovered in the
Gulf of Mexico and OPEC added new capacity. The price
160 50
of the far-dated NYMEX oil futures contract (currently
for delivery in December 2011) rose about $10 per bar­
140 40
rel during 2004, possibly reflecting expectations of greater
120 Oil 30 oil demand in Asian emerging-market economies. The
far-dated futures contract averaged about $38 per barrel
100 20 in January 2005, while the spot price of WTI averaged
about $48 per barrel.
Nonfuel
80 10
The Financial Account
2001 2002 2003 2004 2005
In 2004, the U.S. current account deficit was financed
NOTE. The data are monthly and extend through January 2005. The oil
once again largely by foreign purchases of U.S. bonds.
price is the spot price of West Texas intermediate crude oil. The price of
nonfuel commodities is an index of forty-five primary-commodity prices. Foreign official inflows picked up further last year and
SOURCE. For oil, Wall Street Journal; for nonfuel commodities, Inter- were especially strong in the first quarter, reflecting siz­
national Monetary Fund.
able bond purchases by Asian central banks. Private for­
eign purchases of U.S. bonds rebounded in 2004 from a
diminishing the cartel’s estimated spare capacity to his­
slight decline in 2003, with especially large purchases
torically low levels. Increased OPEC production damped
coming late in the fourth quarter. In contrast, foreign
particularly the rise in prices of heavier, more sulfurous
demand for U.S. equities weakened further in 2004,
grades of crude oil but had less effect on prices of lighter
although this also picked up late in the year. Net pur­
grades like WTI. Supply disruptions also played a role in
chases of foreign securities by U.S. investors remained
the run-up of oil prices. In October, Hurricane Ivan
strong in 2004, with most of the strength coming in the
extensively damaged oil and gas production facilities in
second half of the year.
the Gulf of Mexico, boosting the price of WTI relative to
U.S. direct investment abroad continued at a strong
other grades of crude oil. Sabotage of production and
pace, as reinvested earnings remained sizable. Direct
distribution facilities in Iraq hindered oil exports from
investment into the United States rebounded in the first
that country, which remain below pre-war levels. In
three quarters of 2004 from its anemic pace in 2003; glo­
Nigeria, ethnic violence and community protests shut
bal mergers and acquisitions revived, and reinvested earn­
ings picked up. Overall, net direct investment outflows
continued over the first three quarters of 2004 but at a
Prices of major nonfuel commodities
lower pace than in 2003.
January 2001 = 100
U.S. net financial inflows
145
Metals Billions of dollars
Beverages
130 Official
200
Private
175
115
Food 150
Agricultural 100 125
raw materials
100
85 75
50
25
2001 2002 2003 2004 2005 +
NOTE. The data are monthly and extend through January 2005. The metals
0_
category includes aluminum, copper, and iron ore; food includes cereals, 25
vegetable oils and protein meals, seafood, and meat; agricultural raw
materials consists of timber, cotton, wool, rubber, and hides; beverages 2001 2002 2003 2004
consists of coffee, cocoa beans, and tea.
SOURCE. International Monetary Fund. SOURCE. Department of Commerce.
16 Monetary Policy Report to the Congress February 2005
U.S. net international securities transactions May; net hiring then dropped back to subpar rates of about
100,000 per month in June through September. In the four
Billions of dollars months since then, increases in private payrolls have
Net private foreign purchases of U.S. securities averaged 165,000 per month.
175
The improved pace of hiring was widespread, as all
Bonds
150 major industry groups contributed to faster employment
Equities
125 growth relative to that of the latter part of 2003. The larg­
est gains were in professional and business services and
100
health services. The construction sector also posted sub­
75
stantial gains. In the manufacturing sector—where
50 employment had declined almost continuously since early
25 2000—payrolls increased in the spring when overall
+ employment was rising sharply but were about unchanged,
0_
on net, over the second half of the year. Employment gains
2001 2002 2003 2004 in retail trade and in food services were also brisk over
the first half of the year but tapered off in the second
SOURCE. Department of Commerce and the Federal Reserve Bank of New
York. half. Meanwhile, state and local governments added sub­
stantially to their payrolls last year, especially for educa­
Net inflows of portfolio capital exceeded net outflows tion, but civilian employment in the federal government
of direct investment and represented the financial coun­ edged lower.
terpart to the U.S. current account deficit. These net The unemployment rate fell from near 6 percent in
financial inflows imply a further decline in the U.S. net late 2003 to less than 51/
2
percent by late last year; job­
international investment position, which began 2004 at lessness fell further in January 2005, to 51/
4
percent. The
a reported level of negative $2.4 trillion (22 percent of decline in the unemployment rate over the past year
GDP). reflected both the pickup in hiring and a labor force par­
ticipation rate that remained surprisingly low. From 2001
through 2003, the participation rate declined by more than
The Labor Market would have been predicted on the basis of past relation-
ships with indicators of labor demand, and in 2004, when
Employment and Unemployment
the pace of hiring increased, the participation rate lev­
The labor market improved notably in 2004. Private pay- eled off but failed to rise. These considerations suggest
rolls, which began to post sustained increases in late 2003, that there may be a persistent component to the recent
rose an average of 170,000 per month last year. Progress softness in participation. However, participation had been
was not steady over the course of the year, however. quite strong through 2000, when the labor market was
Employment growth stepped up sharply in the spring to a extremely tight, and the fact that participation turned down
pace of almost 300,000 per month in March, April, and
Civilian unemployment rate
Net change in payroll employment
Percent
Thousands of jobs, monthly average
Private nonfarm
Jan. 300
9
200
100 6
+
0_
3
100
200
1975 1985 1995 2005
1999 2001 2003 2005
NOTE. The data are monthly and extend through January 2005.
SOURCE. Department of Labor, Bureau of Labor Statistics. SOURCE. Department of Labor, Bureau of Labor Statistics.
Board of Governors of the Federal Reserve System 17
Labor force participation rate gains seem to have been related to more effective use of
capital equipment that had been acquired earlier and to
Percent one-time organizational innovations induced by firms’
earlier reluctance to commit to increased hiring. Still, last
year’s 21/
2
percent increase in productivity was impres­
67 sive by long-run standards: It was in line with the pace of
the late 1990s and well above rates that had prevailed
during the preceding two decades.
64 Increases in hourly labor compensation remained mod­
erate last year. As measured by the employment cost in­
dex (ECI), which is based on a quarterly survey from the
61 Bureau of Labor Statistics, hourly compensation in pri­
vate nonfarm businesses increased 33/
4
percent in 2004,
a bit less than in 2003. An alternative measure is com­
1975 1985 1995 2005 pensation per hour in the nonfarm business sector as
derived from compensation data in the NIPAs. This mea­
NOTE. The data are monthly and extend through January 2005.
SOURCE. Department of Labor, Bureau of Labor Statistics. sure of hourly compensation rose 31/
2
percent last year,
an increase similar to that in the ECI but substantially
at the same time that labor demand weakened suggests less than the 51/
2
percent rise in 2003.
that at least some of the recent low participation is cycli­ As has been the case for several years, the cost of
cal. To the extent that some of this low participation proves employee benefits rose considerably more than did wages
to be transitory, the resumption of more-rapid labor force and salaries last year. The benefits component of the ECI
growth will limit the speed at which employment gains increased nearly 7 percent, while the wages and salaries
further push down the unemployment rate. component posted a much more moderate 3 percent
increase. The rise in hourly wages and salaries was about
the same as increases in the preceding two years; although
Productivity and Labor Costs probably boosted by last year’s higher rate of price infla­
tion, wages were likely held down by the continued,
Labor productivity rose solidly again last year. Output
though diminishing, labor market slack and also by
per hour in the nonfarm business sector increased an
employers’ attempts to offset continued large increases
estimated 21/
2
percent over the year. This increase was
in benefits costs. Health insurance costs continued to rise
somewhat below the outsized 4 percent average pace of
rapidly. As measured by the ECI, employers’ costs of
increase from 2001 through 2003. Those earlier huge pro­
health insurance, which account for about 6 percent of
ductivity gains were not associated with especially large
accumulations of new capital equipment, as had been the
case during the late 1990s; instead, to a large degree, the Measures of change in hourly compensation
Percent
Change in output per hour
Percent, annual rate 8
Nonfarm compensation per hour
5 6
4
4
3 Employment
cost index 2
2
1996 1998 2000 2002 2004
1
NOTE. The data are quarterly and extend through 2004:Q4. For nonfarm
compensation, change is over four quarters; for the employment cost index
1948–73 1973–95 1995–2000 2002 2004 (ECI), change is over the twelve months ending in the last month of each
quarter. Nonfarm compensation is for the nonfarm business sector; the ECI is
NOTE. Nonfarm business sector. for private industry excluding farm and household workers.
SOURCE. Department of Labor, Bureau of Labor Statistics. SOURCE. Department of Labor, Bureau of Labor Statistics.
18 Monetary Policy Report to the Congress February 2005
overall compensation costs, rose 7 percent last year after Change in PCE prices excluding food and energy
having increased more than 10 percent per year in 2002
and 2003. Percent, annual rate
Prices
2
Overall consumer prices rose notably more in 2004 than
they did in 2003, and the sharp increase in energy prices
accounted for much of the step-up. The chain-type price
index for personal consumption expenditures (PCE) rose
1
21/
2
percent last year, compared with an increase of
13/
4
percent in 2003. The increase in PCE prices exclud­
ing food and energy was considerably smaller—only
11/
2
percent, up a little more than ¼ percentage point from
1998 2000 2002 2004
the increase in 2003. Inflation as measured by the
market-based component of core PCE prices—which
SOURCE. Department of Labor, Bureau of Labor Statistics.
excludes a collection of erratic prices that are unobserv­
able from market transactions and which the Bureau of
the energy price increase was attributable to the higher
Economic Analysis began to publish early last year—was
prices of imported oil, which are excluded from GDP
in line with overall core PCE inflation last year. The core
because they are not part of domestic production. GDP
consumer price index (CPI) rose about 2 percent last year
prices increased 21/
2
percent last year, 3/
4
percentage point
after having increased 11/
4
percent in 2003. (The CPI dif­
faster than in 2003. In addition to the rise in PCE prices
fers from PCE prices in a number of respects, but one
(excluding the influence of imported oil), GDP prices
factor that boosted CPI inflation relative to PCE infla­
were affected by a sizable increase in construction prices
tion last year was a difference in the way the two indexes
for residential and nonresidential structures.
measure the prices of medical services, especially physi­
The jump in consumer energy prices in 2004 was
cians’ services, which rose much more rapidly in the CPI
driven by the run-up in crude oil prices. The prices of
than in the PCE index.) The rise in core consumer prices
both gasoline and fuel oil increased approximately
was largest in the early months of 2004: Core PCE prices
30 percent over the year, and higher oil costs accounted
increased at an annual rate of nearly 2 percent over the
for the bulk of the increase. Prices of natural gas, which
first half of the year and then decelerated to a 11/
4
percent
can often substitute for fuel oil in the industrial sector,
rate of increase in the second half.
rose notably as well last year despite the restraining in­
The price index for GDP was less affected by last year’s
fluence of ample inventories. Electricity prices, which
rise in energy prices than was the PCE measure; much of
tend to reflect fuel costs with a lag, also moved higher
through most of the year but dropped back some near
year-end.
Change in consumer prices
Consumer food prices rose around 3 percent for a sec­
ond consecutive year in 2004. Exports of beef dropped
Percent
Consumer price index
Chain-type price index for PCE Alternative measures of price change
Percent
4 Price measure 2002 2003 2004
Chain-type
Gross domestic product ............................ 1.6 1.7 2.4
Gross domestic purchases ......................... 1.8 1.8 2.9
Personal consumption expenditures ......... 1.8 1.7 2.5
2 Excluding food and energy ................... 1.5 1.2 1.6
Market-based PCE excluding
food and energy ................................. 1.4 1.0 1.6
Fixed-weight
Consumer price index ............................... 2.2 1.9 3.4
Excluding food and energy ................... 2.0 1.2 2.1
1998 2000 2002 2004
NOTE. Changes are based on quarterly averages of seasonally adjusted data.
SOURCE. For consumer price index, Department of Labor, Bureau of Labor SOURCE. For chain-type measures, Department of Commerce, Bureau of Eco­
Statistics; for chain-type measure, Department of Commerce, Bureau of nomic Analysis; for fixed-weight measures, Department of Labor, Bureau of Labor
Economic Analysis. Statistics.
Board of Governors of the Federal Reserve System 19
sharply last year when most of the largest importing coun­ TIPS-based inflation compensation
tries placed restrictions on U.S. beef after a case of mad
cow disease was discovered. Nevertheless, domestic Percentage points
demand was sufficiently strong to support consumer meat
prices last year. Fruit and vegetable prices trended side- 3.5
ways through most of the year but then rose sharply in Five-year, five-year ahead
3
the fall because of crop damage associated with the
series of hurricanes that hit the Southeast in August and 2.5
September. In addition, prices for food away from home,
which are driven more by labor costs than by raw food 2
prices, increased more rapidly last year than in 2003.
1.5
Core consumer prices were influenced by a variety of
forces last year. Price increases were likely restrained by Five-year 1
continuing slack in labor markets and in some product
markets, but businesses faced considerable pressure from 2002 2003 2004 2005
several sources of increased costs. First, the indirect
NOTE. The data are daily and extend through February 9, 2005. Based on a
effects of the large jump in energy prices fed through to comparison of the yield curve for Treasury Inflation-Protected Securities
(TIPS) to the nominal off-the-run Treasury yield curve.
businesses throughout the economy and were especially
important for firms in energy-intensive industries, such
declines in 2003. In particular, the prices of new motor
as those that produce plastics and fertilizers. Second,
vehicles leveled off after falling notably in 2003, and the
prices were up sharply for a number of other industrial
prices of used vehicles reversed some of their sharp 2003
commodities, including lumber and a variety of metals.
declines. Prices of non-energy PCE services rose about
These price increases reflected strengthening economic
2 percent in 2004—a smaller increase than in 2003.
activity abroad as well as in the United States. Although
Last year’s rise in inflation showed through to short-
these non-oil commodities represent a small part of busi­
term measures of expected inflation, but longer-term mea­
nesses’ overall costs, some businesses likely felt the pinch
sures remained stable. According to the Michigan SRC,
of sustained price increases in these areas. Third, the
households’ median expectations for inflation over the
declining exchange value of the dollar boosted import
next year moved up considerably in the spring as infla­
prices, including those of many inputs to production.
tion was rising, but then they eased back and ended the
Finally, the deceleration in labor productivity boosted unit
year near 3 percent—up from around 21/
2
percent in late
labor costs after two years of declines; nevertheless, last
2003. In contrast, the median expectation for inflation
year’s 1 percent rise in unit labor costs was quite modest.
over the next five to ten years held about steady near
Taken together, these influences left their clearest mark
2¾ percent throughout this period. Inflation compensa­
on the prices of goods rather than services. Core goods
tion as measured by spreads between yields on nominal
prices were about unchanged, on average, last year, but
Treasury securities and inflation-indexed securities—
this period of stability followed a period of unusually large
another indicator of expected inflation, albeit one that is
also influenced by perceptions of inflation risk and per-
Change in unit labor costs haps also by the development of the market for inflation-
indexed debt—showed a similar pattern. Inflation com­
Percent
pensation over the next five years moved up about
1/
2
percentage point during 2004, to 21/
2
percent, while
compensation at the five- to ten-year horizon edged lower,
4 on net, over the year.
2 U.S. Financial Markets
Domestic financial conditions were supportive of eco­
+
0_ nomic growth in 2004. Interest rates on longer-term Trea­
sury securities remained low, corporate risk spreads fell,
and stock prices, on balance, registered gains. These
developments occurred even as market participants
1998 2000 2002 2004
revised up their expectations for the path of the federal
NOTE. Nonfarm business sector.
SOURCE. Department of Labor, Bureau of Labor Statistics. funds rate. At the beginning of 2004, futures market quotes
20 Monetary Policy Report to the Congress February 2005
Interest rates on selected Treasury securities In the statement released after its May meeting, the Com­
mittee indicated that policy accommodation was likely
Percent to be removed at a “measured” pace. At its June meeting,
the Committee raised the target for the federal funds rate
Ten-year 6 from 1 percent to 11/ 4 percent, but it continued to assess
the risks to sustainable growth and to price stability as
5
balanced and reiterated the “measured pace” language.
4 Interest rates across the term structure declined some-
what immediately after the announcement, reportedly
3 because some market participants had expected the
Two-year
FOMC to mention upside risks to growth or inflation in
2
its statement.
Three-month
1 Chairman Greenspan’s congressional testimony in July
on monetary policy, which suggested that recent softness
2002 2003 2004 2005 in consumer spending would likely prove short lived,
sparked a jump in yields on Treasury securities. How-
NOTE. The data are daily and extend through February 9, 2005.
SOURCE. Department of the Treasury. ever, interest rates subsequently moved lower, on bal­
ance, as incoming data pointed to weaker spending and
employment than investors had expected as well as to
implied that investors expected a 13/
4
percent target for
more-subdued core inflation. Apart from the August
the federal funds rate at year-end, 50 basis points below
employment report, which seemed to hint that the
the target actually established at the FOMC meeting in
economy was emerging from its “soft patch,” incoming
December 2004. Consistent with the revision in policy
economic news remained somewhat lackluster through
expectations, yields on two-year Treasury notes increased
the end of the third quarter. However, investors report­
about 11/
4
percentage points in 2004. Yields on longer-
edly viewed FOMC statements and comments by FOMC
dated Treasury securities, however, ended the year
officials as more sanguine on near-term prospects for the
essentially unchanged. Despite the run-up in oil prices,
economy than they had expected. In particular, the
equity prices registered solid gains in 2004 after rising
release of the minutes from the August FOMC meeting,
sharply the year before. Risk spreads on investment-grade
which referenced the probable need for “significant
corporate debt declined a touch, and those on specula­
cumulative tightening,” prompted investors to mark up
tive-grade debt fell more noticeably. Moreover, banks
their expectations for the near-term path of monetary
appreciably eased terms and standards for lending to
policy.
businesses.
Short-term Treasury yields rose a bit further over the
fall in association with actual and expected policy tight­
ening, but long-term Treasury yields were little changed
Interest Rates
on net. Investors’ expectations for the path of monetary
Most market interest rates rose, on balance, over the first policy firmed a bit more in the fourth quarter in response
half of 2004, particularly at shorter maturities. The to higher-than-anticipated inflation and remarks from
FOMC’s decision at its January meeting to shift from a Federal Reserve officials that were reportedly interpreted
statement that monetary policy could remain accommo­ as suggesting that an imminent pause in the tightening
dative for “a considerable period” to an indication that it cycle was unlikely.
could be “patient” in removing policy accommodation As the economic expansion gathered momentum and
prompted a rise in market interest rates. In early Febru­ measures of corporate credit quality improved, investors’
ary and March, yields fell substantially in response to perception of risk seemed to diminish, and their willing­
employment reports that indicated tepid job growth. Prices ness to bear risk apparently increased. Risk spreads on
of federal funds and Eurodollar futures contracts implied investment-grade corporate debt over comparable Trea­
that investors placed only small odds on an increase in suries ended the year slightly below their levels at the
the target funds rate before late 2004 and that they envi­ end of 2003. Spreads of speculative-grade yields declined
sioned only moderate monetary policy tightening there- further after narrowing sharply during 2003.
after. Longer-term interest rates and the expected path In early 2005, market participants boosted their
for the federal funds rate were considerably marked up expectations for the path of the federal funds rate, partly
later in the spring in response to data suggesting a pickup in response to the publication of the minutes of the
in aggregate demand and hiring, readings on core infla­ December FOMC meeting, which investors reportedly
tion that came in above expectations, and rising oil prices. interpreted as pointing to greater concerns about infla-
Board of Governors of the Federal Reserve System 21
Spreads of corporate bond yields over Implied S&P 500 volatility
comparable off-the-run Treasury yields
Percent
Percentage points
10 40
8
30
High-yield
6
20
BBB 4
2 10
AA +
0_
1998 1999 2000 2001 2002 2003 2004 2005
1997 1998 1999 2000 2001 2002 2003 2004 2005
NOTE. The data are daily and extend through February 9, 2005. The series
NOTE. The data are daily and extend through February 9, 2005. The shown is the implied thirty-day volatility of the S&P 500 stock price index as
high-yield index is compared with the five-year Treasury yield, and the BBB calculated from a weighted average of options prices.
and AA indexes are compared with the ten-year Treasury yield. SOURCE. Chicago Board Options Exchange.
SOURCE. Merrill Lynch AA and BBB indexes and Merrill Lynch Master II
high-yield index.
tion than had been expected. Short- and intermediate-term in the second quarter, worries about geopolitical devel­
Treasury yields rose along with expectations for the path opments, and sharply higher oil prices. Stock prices
of monetary policy, but longer-term yields edged lower. dipped early in the second half in response to softer eco­
Yields on investment- and speculative-grade corporate nomic data, further concerns about energy prices, and
bonds largely moved with those on comparable Treasury guidance from corporations that pointed to a less opti­
securities, and hence risk spreads remained at low levels. mistic trajectory for earnings than investors had report­
edly been expecting. However, as oil prices pulled back
toward the end of 2004 and news on the economy
Equity Markets improved, stock prices rebounded to post solid gains for
the year. The increases were led by stocks with compara­
After surging as much as 30 percent in 2003, broad stock
tively small market capitalizations; the Russell 2000
market indexes climbed modestly over the first half of
index climbed 17 percent in 2004 to a record high. The
2004. The boost to equity prices from robust earnings
S&P 500 and the technology-laden Nasdaq advanced
reports and analysts’ upward revisions for future profits
about 9 percent and 81/
2
percent respectively. To date in
during this period was offset in part by rising interest rates
2005, equity prices have edged lower, on balance, as
investors have responded to a rebound in oil prices,
lackluster earnings reports, cautious guidance for future
Stock price indexes
profits, and indications of continued monetary policy
tightening.
January 2, 2003 = 100
Expected volatility implied by options prices for both
170 the Nasdaq 100 and the S&P 500 declined further in
Russell 2000 160 2004 from already low levels. The difference between
the earnings–price ratio and the real ten-year Treasury
150
yield—a crude measure of the premium investors require
140
for holding equity shares—changed little, on balance,
130
remaining close to its average value over the past two
120
decades but above its level during the late 1990s.
Wilshire 5000
110
100
90 Debt, Bank Credit, and M2
2003 2004 2005 The aggregate debt of domestic nonfinancial sectors is
NOTE. The data are daily and extend through February 9, 2005.
estimated to have increased about 73/
4
percent in 2004,
22 Monetary Policy Report to the Congress February 2005
Growth of domestic nonfinancial debt mergers. After adjusting for certain reclassifications of
securities as loans, the growth of consumer loans on banks’
Percent books remained sluggish. Despite reports of increased
competition among banks and nonbank intermediaries,
bank profits were again strong in 2004. Banks experi­
8 enced further improvements in asset quality and, as a
result, reduced their provisions for loan losses.
Total
M2 grew at a pace roughly in line with that of nominal
6 GDP during the first half of 2004. A resurgence of mort­
gage refinancing spurred by the first-quarter decline in
mortgage rates likely boosted liquid deposit growth, as
4 proceeds from refinancing were temporarily held in
deposit accounts pending disbursement to the holders of
mortgage-backed securities. M2 growth slowed in the
second half of the year in response to a drop in mortgage
Percent
refinancing activity and the increased opportunity cost
of holding M2 assets, as returns available on market
15
instruments rose more than those on M2 components. For
Nonfederal 10 example, yields on retail money market mutual funds
moved up more slowly than did short-term market inter­
5
est rates, and assets of money funds accordingly contin­
+
ued to shrink. Small time deposits, which had contracted
0_
over the previous three years, resumed expansion in the
Federal, 5 second half of the year, as their yields began to rise in
held by public
association with the increase in other market rates. Cur­
10
rency grew at its slowest rate since 2000, apparently
reflecting sluggish demand by both domestic and foreign
1990 1992 1994 1996 1998 2000 2002 2004
holders. On balance, M2 growth from the fourth quarter
ear
N
li
O
e
T
r
E
y
.
e
F
ar
o
s
r
,
2
t
0
h
0
e
4
d
,
a
c
t
h
a
a
a
n
r
g
e
e
a
is
n n
fr
u
o
a
m
l a
2
n
0
d
0 3
a
:
r
Q
e
4
c o
to
m
2
p
0
u
0
t
4
ed
:Q
b
3
y
a t
d i
a
v
n
i d
a
i
n
n
n
g
u a
th
l
e
ra
a
te
n
.
n
F
u
o
a
r
l
of 2003 to the fourth quarter of 2004 was about 51/
4
per-
flow for a given year by the level at the end of the preceding year. The total cent. The velocity of M2 rose 1 percent, on net, roughly
consists of nonfederal debt and federal debt held by the public. Nonfederal in line with the historical relationships among money,
debt consists of the outstanding credit market debt of state and local gov­
ernments, households, nonprofit organizations, and nonfinancial businesses. income, and opportunity cost.
Federal debt held by the public excludes securities held as investments of
federal government accounts.
somewhat faster than nominal income but a bit slower
M2 growth rate
than the pace set the year before. Household and federal
debt expanded rapidly. Borrowing by nonfinancial busi­
Percent
nesses was moderate, although it picked up in the fourth
quarter.
10
Commercial bank credit rose about 9 percent in 2004,
a larger advance than in the previous year. Expansion of
8
mortgage and home equity loans on banks’ books
remained strong, as activity in the housing market stayed
6
robust while mortgage originations shifted somewhat
toward adjustable-rate products. After several years of 4
runoffs, business loans began to grow in the second quar­
ter of the year. According to survey evidence, commer­ 2
cial banks eased terms and standards on business loans
as the economic outlook improved and competition from
1990 1992 1994 1996 1998 2000 2002 2004
other banks and nonbank lenders intensified. Also, banks
reported a pickup in demand for business loans that was
NOTE. The data are annual and extend through 2004. M2 consists of
currency, traveler’s checks, demand deposits, other checkable deposits,
said to be driven by customers’ needs to fund rising savings deposits (including money market deposit accounts),
small-denomination time deposits, and balances in retail money market
accounts receivable, inventories, capital expenditures, and
funds.
Board of Governors of the Federal Reserve System 23
International Developments Equity indexes in selected foreign industrial countries
Foreign economic activity expanded in 2004 at a faster
Week ending January 4, 2002 = 100
pace than in the preceding three years. The pickup in
growth was widespread—global manufacturing and trade
Canada 115
rebounded across industrial and emerging economies, in
Japan
part because of strong demand from the United States
100
and China. In the second half of the year, trade and for­
eign GDP growth slowed, partly as a result of higher oil United Kingdom
85
prices and the appreciation of some foreign currencies
against the dollar. The run-up in oil prices and other com­
70
modity prices contributed to higher, though still moder­
ate, inflation across industrial and emerging economies. Euro area 55
Monetary policy in many foreign economies tightened
over the course of 2004. Citing high rates of capacity
2002 2003 2004 2005
utilization and mounting inflationary pressures, the Bank
of England raised its target interest rate 100 basis points
NOTE. The data are weekly. The last observation for each series is the
average of trading days through February 9, 2005.
but has been on hold since August amid signs that hous­ SOURCE. Bloomberg L.P.
ing prices and consumer spending are cooling. After cut­
ting official interest rates earlier in the year, the Bank of
2004. Rates rose in the second quarter as new data
Canada raised rates in the fall in response to diminishing
(including reports from the United States) that showed
slack in the economy. The Bank of Mexico tightened
faster growth and higher inflation led market participants
policy throughout the year to resist rising inflation, and
to expect more-aggressive monetary tightening. However,
Chinese authorities made monetary policy more restric­
foreign long-term interest rates slipped after midyear,
tive to rein in soaring investment demand. In the euro
when foreign growth slowed and foreign currencies
area and Japan, central banks kept policy interest rates
appreciated against the dollar. Over the first half of the
unchanged in 2004.
year, spreads on internationally issued sovereign debt of
Foreign equity price indexes recorded moderate net
emerging-market economies over U.S. Treasuries moved
gains last year after larger increases in 2003. Equity mar­
up somewhat from low levels, but spreads more than
kets started the year strong, but prices declined in the
reversed those increases in the second half.
spring as interest rates rose. The run-up in oil prices
The path of the exchange rate was uneven over the
between July and October appeared to weigh on foreign
course of 2004. The dollar rose slightly in the first half of
equity prices, but the subsequent decline in oil prices
helped support a rise in equity prices late in the year.
Foreign long-term interest rates declined, on net, during
Spread on internationally issued sovereign debt
of emerging-market economies
Official interest rates in selected foreign industrial countries
Percentage points
Percent
10
6
5
8
United Kingdom
4
Canada 3 6
2
Euro area
4
1
Japan +
0_
2002 2003 2004 2005
2001 2002 2003 2004 2005 NOTE. The data are weekly averages. The last observation is the average of
trading days through February 9, 2005. The series shown is the spread of the
NOTE. The data are as of month-end; the last observation for each series is yield of certain dollar-denominated sovereign debt instruments of emerg­
the average of trading days through February 9, 2005. The data shown are the ing-market economies over U.S. Treasury securities; over the period shown,
call money rate for Japan, the overnight rate for Canada, the refinancing rate the index encompassed nineteen countries.
for the euro area, and the repurchase rate for the United Kingdom. SOURCE. J.P. Morgan Emerging Market Bond Index Plus (EMBI+).
24 Monetary Policy Report to the Congress February 2005
U.S. dollar nominal exchange rate, broad index mentary focusing on the positive differential between U.S.
economic growth and that in Europe and Japan.
January 2001 = 100
105 Industrial Economies
After increasing strongly in the first quarter, Japanese
100 GDP growth stagnated in the remainder of 2004. Growth
in exports and business investment slowed over the year,
95 and government investment contracted. However, corpo­
rate profits and balance sheets improved, and labor mar­
ket conditions also brightened, with the job-offers-to-ap­
90
plicants ratio rising to a twelve-year high. Consumer
prices continued to decline in 2004, though only slightly.
2001 2002 2003 2004 2005 In contrast, higher commodity prices helped push twelve-
month wholesale price inflation up to 2 percent late in
NOTE. The data are monthly and are in foreign currency units per dollar.
The last observation is the average of trading days through February 9, 2005. the year, its highest rate since 1990. The yield on the ten-
The broad index is a weighted average of the foreign exchange values of the
U.S. dollar against the currencies of a large group of major U.S. trading year bellwether government bond rose from its June 2003
partners. The index weights, which change over time, are derived from U.S. record low of about 1/
2
percent to nearly 2 percent in mid-
export shares and from U.S. and foreign import shares.
year before retreating to about 11/
2
percent recently.
After making substantial sales of yen for dollars in the
the year on perceptions that monetary policy would tighten
first quarter, Japanese authorities ceased intervention in
more quickly in the United States than abroad. Begin­
mid-March and remained on the sidelines even as the yen
ning in September, however, the dollar resumed the
appreciated significantly against the dollar in the fall.
depreciation that had started in 2002, as market partici­
Economic conditions in the euro area firmed during
pants focused on the financing implications of the large
the first half of 2004 but weakened in the second half.
and growing U.S. current account deficit. In 2004, the
Private consumption and investment spending continued
dollar depreciated about 7 percent, on net, against the
to rise, but export growth slowed after midyear. German
euro, the U.K. pound, and the Canadian dollar. The dol­
GDP growth slowed to a crawl in the second half, as
lar declined 4 percent, on net, against the Japanese yen
German consumer spending remained anemic, held down
and 13 percent against the Korean won, but some other
by a weak labor market and low consumer confidence. In
Asian central banks, most notably the People’s Bank of
contrast, French GDP growth was strong in the fourth
China, kept their currencies stable against the dollar. So
quarter. The euro-area unemployment rate has been near
far in 2005, the dollar has rebounded, with market com-
9 percent since rising to that level in early 2003. Infla­
tion for the euro area remained just above the European
U.S. dollar exchange rate against Central Bank’s medium-term goal of less than, but close
selected major currencies to, 2 percent.
With the exception of a slowdown in the third quarter,
Week ending January 4, 2002 = 100 economic expansion in the United Kingdom stayed strong
during 2004, largely because of the brisk growth of con­
Canadian
sumption and government spending. Labor markets
dollar 100
remained tight in 2004; the unemployment rate ticked
down to its lowest level in almost three decades, and
90 labor earnings posted solid gains. Consumer price infla­
Japanese yen
tion over the twelve months ending in December was
Euro 80 11/ 2 percent, below the central bank’s official target rate
of 2 percent. Housing price rises slowed sharply from
rapid rates and were muted during the second half of 2004.
70
U.K. Household net mortgage borrowing declined to a level
pound
20 percent below its 2003 peak.
2002 2003 2004 2005 The Canadian economy expanded at a healthy pace
NOTE. The data are weekly and are in foreign currency units per dollar. throughout 2004. Sizable gains in consumption and
The last observation for each series is the average of trading days through investment boosted output throughout the year. Export
February 9, 2005.
SOURCE. Bloomberg L.P. growth, supported by demand from the United States, was
Board of Governors of the Federal Reserve System 25
strong in the first half of the year but stagnated in the can government spending to provide stimulus while still
second half as U.S. manufacturing growth slowed and meeting fiscal targets.
the Canadian dollar’s appreciation hurt Canadian trade. In Brazil, economic activity continued to expand
The unemployment rate declined moderately over the robustly in 2004. Domestic demand was supported by
year, and employment posted strong gains. Consumer the monetary loosening that occurred in the second half
price inflation has settled at about 2 percent, the mid- of 2003 and early 2004. Export growth was boosted by
point of the Bank of Canada’s inflation target range, demand for commodities and the recovery in Argentina.
whereas inflation excluding food, energy, and indirect Brazilian asset prices declined through May on expecta­
taxes declined to around 11/
2
percent by year-end. tions that higher global interest rates would make it more
difficult for the Brazilian government to finance its debt,
but stock prices have moved up sharply since May, and
Emerging-Market Economies the currency has appreciated. Concerns over inflation
pressures have prompted the central bank to tighten mon­
Growth of real GDP in China remained very robust in
etary policy since September.
2004, supported by strong domestic demand and exports.
In Argentina, the economic recovery picked up steam
The Chinese government took steps early in the year to
last year, as exports were supported by strong demand
slow investment spending, curbing investment approvals
for commodities. The country continues, however, to
and lending. Investment growth slowed significantly
grapple with difficult structural problems. After more than
but remained rapid. At the same time, indicators of per­
three years in default, the government launched a debt
sonal consumption spending strengthened, and Chinese
swap in January with the goal of restructuring more than
exports and imports continued to soar in 2004. Consumer
$80 billion in defaulted bonds.
price inflation peaked at a twelve-month change of more
than 5 percent in July but has fallen since then to less
than 3 percent, as food prices have moderated. Inflation Equity indexes in selected emerging-market economies
excluding food is only about 1 percent.
Week ending January 3, 2003 = 100
Supported by exports to China, economic growth in
other Asian emerging-market economies was generally
strong in 2004. Economic expansion in Korea remained 255
Argentina
heavily dependent on external demand because high lev­
els of consumer debt continued to weigh on consumption 210
spending. Inflation across emerging Asia, though still Mexico
165
moderate, was pushed up by higher energy prices and
Asian emerging-
strong aggregate demand. market economies
120
The Mexican economy grew rapidly in the first half Brazil
of the year in response to strong demand from the United
75
States. In the third quarter, Mexican GDP growth slowed
somewhat, as manufacturing exports stagnated, but
domestic demand remained buoyant. Increases in energy 2002 2003 2004 2005
and food prices pushed up twelve-month consumer price NOTE. The data are weekly. The last observation for each series is the av­
erage of trading days through February 9, 2005. The Asian emerging-market
inflation to more than 5 percent, above the Bank of
economies are China, Hong Kong, India, Indonesia, Malaysia, Pakistan, the
Mexico’s target range of 2 percent to 4 percent. Mon­ Philippines, Singapore, South Korea, Taiwan, and Thailand; the index weight
for each of these economies is its market capitalization as a share of the
etary policy tightened throughout the year, and inflation group’s total.
began to fall near year-end. Oil revenues boosted the SOURCE. For Argentina, Brazil, Mexico, Bloomberg L.P.; for Asian
emerging-market economies, Morgan Stanley Capital International (MSCI)
Mexican public-sector fiscal surplus and allowed Mexi­ index.'''

cleaned = clean_text(sample, drop_notes=DROP_NOTES)
print("---- CLEANED ----")
print(cleaned)
pieces = chunk_text(cleaned, TARGET_TOKENS, MIN_TOKENS)
print("\n---- CHUNKS ----")
print(len(pieces), "chunk(s); tokens:", [n_tokens(c) for c in pieces])

---- CLEANED ----
For use at 10:00 a.m., EST
Submitted

The Board of Governors is pleased to submit its
Monetary Policy and the Economic Outlook 1
Economic and Financial Developments in 2004 and Early 2005 4
, ing and gains in industrial production dropped back nota bly during the summer. Equity prices and longer-term
interest rates moved lower over this period as well. In the
event, the slowdown in household spending growth proved

short lived. Both hiring and increases in factory output
The year 2004 was marked by continued expansion in stepped up again in the autumn, and these gains were
economic activity and appreciable gains in employment. extended early this year. With profits healthy and finanWith fiscal policy stimulative, monetary policy accomcial conditions still supportive, capital spending
modative, and financial conditions favorable, household increased at a brisk pace throughout the year. Over the
spending remained buoyant and businesses increased final quarter of 2004, s

Full run 

In [14]:
# These four functions sit downstream of clean_text() - once a scraped FED document has had
# its boilerplate and chart junk stripped out, I still need to chunk it up, throw out anything
# that's too thin to be useful, catch duplicates (a lot of the annual/monetary reports repeat
# whole paragraphs year to year), and tag each chunk with which report it actually came from.
# build_corpus() is the orchestrator that ties all of this together and writes the final jsonl.

def quality_ok(text, min_chars, min_alpha):
    # Final quality gate before a chunk gets written to the corpus. Even after clean_text does
    # its aggressive filtering, a chunk can still slip through that's just a fragment of a table
    # or a stray line of chart data that happened to have enough letters to dodge the alpha-ratio
    # filter earlier, so I check it again here at the chunk level.

    # If the chunk is shorter than min_chars it's not really worth keeping, its probably just a
    # heading or a leftover fragment rather than real body text.
    if len(text) < min_chars:
        return False

    # Same idea as the alpha ratio check in clean_text, count the letters and compare to the
    # total length of the chunk.
    letters = sum(c.isalpha() for c in text)
    # max(len(text), 1) is just there so I don't divide by zero on an empty string, shouldnt
    # happen given the min_chars check above but cheap to guard against.
    return letters / max(len(text), 1) >= min_alpha


def dedup_key(text):
    # A lot of these reports reuse the same boilerplate paragraphs across different years, and
    # sometimes the same file gets picked up twice in the scrape, so I need a cheap way to spot
    # duplicate chunks before they end up in the training corpus twice.

    # Collapsing all whitespace down to single spaces and lowercasing everything first, so two
    # chunks that are identical apart from a line break or capitalization still hash to the same
    # key, otherwise trivial formatting differences would let duplicates through.
    # md5 is just a fingerprint here, not doing anything cryptographic, so its fine that its not
    # collision-resistant against an adversary, I just need something fast and consistent.
    return hashlib.md5(re.sub(r"\s+", " ", text).lower().encode("utf-8")).hexdigest()


def source_of(root, path):
    # I want every chunk tagged with which report family it came from (Monetary Policy Report,
    # Financial Stability Report, etc.) so I can filter or analyze by source later on, and that
    # info is encoded in the folder structure rather than the filename itself.

    # Getting the path relative to whichever input_dir its under, then splitting on the OS
    # separator, the first folder in that relative path is the source category.
    rel = os.path.relpath(path, root).split(os.sep)
    # If the file happens to sit directly in the root with no subfolder, there's no category
    # folder to grab, so I just fall back to the root directory's own name instead.
    return rel[0] if len(rel) > 1 else os.path.basename(os.path.normpath(root))


def build_corpus(input_dirs, out_path, target, min_tokens, min_chars, min_alpha, drop_notes):
    # This is the main driver, walks every input dir for the scraped .txt files, cleans each one,
    # chunks it, filters out the low quality/duplicate chunks, and writes everything that survives
    # out to a single jsonl file that I can actually feed into whatever comes next.

    # Recursively glob every input dir for .txt files, keeping the root alongside each path so
    # source_of() further down knows which root a given file belongs to.
    files = []
    for root in input_dirs:
        files += [(root, p) for p in sorted(glob.glob(os.path.join(root, "**", "*.txt"),
                  recursive=True))]
    # Bailing out early with a helpful message rather than silently writing an empty file, this
    # has saved me a few times when INPUT_DIRS in the Config cell was pointed at the wrong place.
    if not files:
        print("No .txt files found under:", input_dirs, "\n-> check INPUT_DIRS in the Config cell.")
        return 0

    # Making sure the output folder actually exists before I try to open a file inside it.
    os.makedirs(os.path.dirname(out_path) or ".", exist_ok=True)
    # seen accumulates dedup keys across ALL files, not just within a single file, since the
    # repeated boilerplate problem shows up across different reports, not just within one.
    seen = set(); n_chunks = n_dups = n_low = 0
    with open(out_path, "w", encoding="utf-8") as out:
        for root, path in files:
            # errors="replace" so a stray bad byte in one scraped file doesn't kill the whole run.
            with open(path, encoding="utf-8", errors="replace") as f:
                cleaned = clean_text(f.read(), drop_notes=drop_notes)
            src, kept = source_of(root, path), 0
            for i, chunk in enumerate(chunk_text(cleaned, target, min_tokens)):
                # Skip anything too thin/numeric to be worth keeping.
                if not quality_ok(chunk, min_chars, min_alpha):
                    n_low += 1; continue
                # Skip anything I've already seen, either from this file or an earlier one.
                k = dedup_key(chunk)
                if k in seen:
                    n_dups += 1; continue
                seen.add(k)
                # Writing one json object per line (jsonl) rather than one big json array, so
                # I can stream through this later without loading the whole corpus into memory.
                out.write(json.dumps({
                    "text": chunk, "source": src,
                    "path": os.path.relpath(path), "chunk": i,
                    "n_tokens": n_tokens(chunk),
                }, ensure_ascii=False) + "\n")
                n_chunks += 1; kept += 1
            # Per-file progress line, mainly so I can watch it work through a big batch of reports
            # and sanity check that a given file actually produced chunks.
            print(f"{path}  [{src}]  +{kept}")
    # Summary stats at the end, mostly so I can eyeball whether the dedup/quality thresholds are
    # too aggressive or too lax without having to go dig through the output file myself.
    print(f"\nfiles={len(files)}  chunks={n_chunks}  dups_skipped={n_dups}  "
          f"low_quality_skipped={n_low}\n-> {out_path}")
    return n_chunks

In [15]:
build_corpus(INPUT_DIRS, OUT_PATH, TARGET_TOKENS, MIN_TOKENS,
              MIN_CHARS, MIN_ALPHA, DROP_NOTES)

/Users/lorenzouberti/Desktop/BANQUE DE FRANCE /Data Folders/FED BRANCH/POLICY DECISIONS/txt/fed_FOMC_remaining_sentences.txt  [txt]  +274

files=1  chunks=274  dups_skipped=0  low_quality_skipped=0
-> fed_remaining_statements_unlabeled.jsonl


274

In [16]:
import statistics
from collections import Counter

recs = [json.loads(l) for l in open(OUT_PATH, encoding="utf-8")]
toks = [r["n_tokens"] for r in recs]
print(len(recs), "records")
print("tokens/chunk  min", min(toks), " median", int(statistics.median(toks)), " max", max(toks))
print("by source:", dict(Counter(r["source"] for r in recs)))

274 records
tokens/chunk  min 733  median 2025  max 2045
by source: {'txt': 274}


In [17]:
print("total tokens:", sum(toks))

total tokens: 552138


In [18]:
print(OUT_PATH)

fed_remaining_statements_unlabeled.jsonl


Token Count

In [19]:
data_dir = Path("data")  # adjust if your notebook's cwd isn't the project root

totals = Counter()
grand_total = 0
grand_records = 0

for path in sorted(data_dir.glob("*.jsonl")):
    n_tok = 0
    n_rec = 0
    with open(path, encoding="utf-8") as f:
        for line in f:
            rec = json.loads(line)
            n_tok += rec["n_tokens"]
            n_rec += 1
    totals[path.name] = n_tok
    grand_total += n_tok
    grand_records += n_rec
    print(f"{path.name:45s}  {n_rec:7,d} chunks  {n_tok:12,d} tokens")

print("-" * 70)
print(f"{'TOTAL':45s}  {grand_records:7,d} chunks  {grand_total:12,d} tokens")

----------------------------------------------------------------------
TOTAL                                                0 chunks             0 tokens
